# v3 — voice + punctuation: the first interaction

**Variant `v3`** of the five-submission study described in
`docs/HYPOTHESIS.md` §9. This notebook is standalone: the `ntl` package is
inlined in the first cell, so it runs on Kaggle with no install and no
internet.

## Hypothesis

Neither block works alone: voice alone is 0.4386 and punctuation alone 0.4327, both neutral. The bet is that they are complementary — punctuation gives the classifier a clean cue, voice facets give it the register that makes the cue usable.

## Expected

0.4730 at 3 folds (+0.027 over control).

## What this notebook does

1. loads the competition train set and, if attached, the public in-style pool;
2. scores this exact configuration with **prompt-grouped** cross-validation
   (never random CV — that measures a different, much harder task);
3. refits on everything and writes `/kaggle/working/submission.parquet`;
4. prints a one-line `TRACKER_JSON` record and appends it to
   `/kaggle/working/tracker.csv`.

Run all five, then compare local CV against the public leaderboard. The
interesting quantity is the **gap**, not any single score.

In [1]:
# ==== ntl package (INLINED — auto-generated, do not edit) ====
# Source of truth: src/ntl/*.py
# Regenerate with:  uv run python scripts/sync_notebook.py
# Makes this notebook self-contained for Kaggle (no pip install, no internet).
import sys as _sys
import types as _types

_NTL_MARKER = "# %% module: "

_NTL_SOURCES = r'''

# %% module: config
"""Central configuration: paths, labels, constants.

Paths resolve transparently between local dev and the Kaggle code-competition
runtime:

* locally:    ``data/raw`` inside the repo (override with ``NTL_DATA_DIR``)
* on Kaggle:  discovered by :func:`competition_dirs`, which searches the
  whole ``/kaggle/input`` mount rather than assuming one layout
"""

from __future__ import annotations

import os
from pathlib import Path

RANDOM_STATE = 42

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------


def _find_project_root() -> Path:
    """Best-effort project root, robust to being exec'd from an inlined cell.

    Searches for the nearest ancestor of ``__file__`` (when present) or the
    current working directory that contains a ``pyproject.toml``; falls back
    to the cwd.
    """
    starts: list[Path] = []
    try:
        starts.append(Path(__file__).resolve().parent)
    except NameError:  # inlined cell: no __file__
        pass
    starts.append(Path.cwd())
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / "pyproject.toml").exists():
                return candidate
    return Path.cwd()


PROJECT_ROOT = _find_project_root()
DATA_DIR = Path(os.environ.get("NTL_DATA_DIR", PROJECT_ROOT / "data" / "raw"))
#: Root of everything Kaggle mounts for a notebook.
KAGGLE_INPUT = Path("/kaggle/input")

#: Name of the competition directory inside the Kaggle mount.
COMPETITION_NAME = "name-that-llm"

OUTPUT_DIR = PROJECT_ROOT / "outputs"


#: Files that identify the competition's own data directory.
COMPETITION_FILES = ("train.csv", "test.csv", "sample_submission.parquet")


def _dir_data_score(path: Path) -> int:
    """How many of :data:`COMPETITION_FILES` live directly in *path*."""
    return sum(1 for f in COMPETITION_FILES if (path / f).exists())


def _walk_for_data_dirs(root: Path, max_depth: int = 5) -> list[Path]:
    """Every directory under *root* that directly holds a competition file."""
    found: list[Path] = []
    root = Path(root)
    if not root.exists():
        return found
    stack: list[tuple[Path, int]] = [(root, 0)]
    while stack:
        current, depth = stack.pop()
        if _dir_data_score(current) > 0:
            found.append(current)
        if depth >= max_depth:
            continue
        try:
            children = sorted(p for p in current.iterdir() if p.is_dir())
        except OSError:
            continue
        stack.extend((child, depth + 1) for child in children)
    return found


def competition_dirs() -> list[Path]:
    """Locate the competition's own data anywhere inside the Kaggle mount.

    Kaggle has used several layouts for this, and they do not all appear in
    the documentation:

    * ``/kaggle/input/competitions/name-that-llm/``
    * ``/kaggle/input/name-that-llm/``
    * ``/kaggle/input/datasets/<owner>/name-that-llm/``

    Rather than guess, we look for the directories that actually hold the
    competition's files and rank them.  The real competition directory holds
    ``train.csv`` *and* ``test.csv`` *and* ``sample_submission.parquet``; a
    public passage pool holds none of them, so the ranking cannot be fooled
    by the datasets you attach alongside the competition.

    Returns a list, best candidate first.  Empty outside the Kaggle runtime.
    """
    known = [
        KAGGLE_INPUT / "competitions" / COMPETITION_NAME,
        KAGGLE_INPUT / COMPETITION_NAME,
        KAGGLE_INPUT / "datasets" / COMPETITION_NAME,
    ]
    comps = KAGGLE_INPUT / "competitions"
    if comps.is_dir():
        known.extend(sorted(p for p in comps.iterdir() if p.is_dir()))

    candidates = [p for p in known if p.is_dir()]
    for found in _walk_for_data_dirs(KAGGLE_INPUT):
        if found not in candidates:
            candidates.append(found)

    def rank(path: Path) -> tuple:
        return (
            -_dir_data_score(path),  # most competition files first
            path.name != COMPETITION_NAME,  # then a directory named exactly so
            "competitions" not in path.parts,  # then one under competitions/
            len(path.parts),  # then the shallowest
            str(path),
        )

    return sorted(dict.fromkeys(candidates), key=rank)


def mount_summary(max_depth: int = 3, max_entries: int = 12) -> list[str]:
    """A readable dump of the Kaggle mount — used in error messages.

    Shows directories and their immediate contents, bounded in both depth
    and width so a large attached dataset cannot flood the output.
    """
    root = KAGGLE_INPUT
    if not root.exists():
        return [f"(no Kaggle mount at {root})"]
    lines: list[str] = []
    stack: list[tuple[Path, int]] = [(root, 0)]
    while stack:
        current, depth = stack.pop()
        try:
            entries = sorted(current.iterdir())
        except OSError:
            continue
        for entry in entries:
            if entry.is_dir():
                lines.append("  " * depth + f"{entry.name}/")
                if depth < max_depth:
                    stack.append((entry, depth + 1))
            else:
                lines.append("  " * depth + entry.name)
        if len(entries) > max_entries:
            lines.append("  " * depth + f"... (+{len(entries) - max_entries} more)")
    return lines


def resolve_path(filename: str) -> Path:
    """Return the first existing path for *filename* across known data dirs.

    Checks ``$NTL_DATA_DIR`` (or ``data/raw``) first, then the Kaggle input
    mount. Raises a FileNotFoundError with a helpful message otherwise.
    """
    candidates = [DATA_DIR / filename]
    candidates.extend(base / filename for base in competition_dirs())
    for candidate in candidates:
        if candidate.exists():
            return candidate
    lines = [f"Could not find {filename!r}. Tried:"]
    lines.extend(f"  {c}" for c in candidates)
    lines.append("The Kaggle mount looks like this:")
    lines.extend(f"  {line}" for line in mount_summary())
    lines.append("")
    lines.append(
        "If the competition data is not attached, open the notebook from the "
        "competition page so Kaggle mounts it, or download the data with:"
    )
    lines.append("  uv run kaggle competitions download -c name-that-llm -p data/raw")
    lines.append("(requires accepting the competition rules and a kaggle.json API token)")
    raise FileNotFoundError(chr(10).join(lines))


# NOTE: file paths are resolved lazily via resolve_path() in data.py so the
# package imports cleanly even before the data is downloaded.


# ---------------------------------------------------------------------------
# Labels — the 18 valid model names
# ---------------------------------------------------------------------------
LABELS: list[str] = [
    # OpenAI
    "gpt-5.5",
    "gpt-5.6-luna",
    "gpt-5.6-sol",
    "gpt-5.6-terra",
    "gpt-6-astra",
    # Google
    "gemini-3.1-pro-preview",
    "gemini-3.6-flash",
    "gemini-3.7-flash",
    "gemini-3.8-flash",
    # Anthropic
    "claude-fable-5",
    "claude-opus-4-6",
    "claude-opus-4-8",
    "claude-opus-5",
    "claude-sonnet-5",
    # Qwen
    "qwen3.8-max",
    # ZAI
    "glm-5.3",
    "glm-5.3-flash",
    # Moonshot AI
    "kimi-k3",
]

PROVIDERS: dict[str, str] = {
    "gpt-5.5": "OpenAI",
    "gpt-5.6-luna": "OpenAI",
    "gpt-5.6-sol": "OpenAI",
    "gpt-5.6-terra": "OpenAI",
    "gpt-6-astra": "OpenAI",
    "gemini-3.1-pro-preview": "Google",
    "gemini-3.6-flash": "Google",
    "gemini-3.7-flash": "Google",
    "gemini-3.8-flash": "Google",
    "claude-fable-5": "Anthropic",
    "claude-opus-4-6": "Anthropic",
    "claude-opus-4-8": "Anthropic",
    "claude-opus-5": "Anthropic",
    "claude-sonnet-5": "Anthropic",
    "qwen3.8-max": "Qwen",
    "glm-5.3": "ZAI",
    "glm-5.3-flash": "ZAI",
    "kimi-k3": "Moonshot AI",
}

# Column names shared across train/test
ID_COL = "id"
PROMPT_COL = "prompt_id"
TEXT_COL = "text"
TARGET_COL = "model"

# Competition description: passages are 40-250 words
MIN_WORDS, MAX_WORDS = 40, 250
N_MODELS = len(LABELS)
N_PROMPTS = 150
# %% module: data
"""Data loading for the Name That LLM competition.

Works both locally (``data/raw``) and on Kaggle (``/kaggle/input/...``) —
see :func:`ntl.config.resolve_path`.
"""

from __future__ import annotations

from collections.abc import Sequence
from pathlib import Path

import numpy as np
import pandas as pd

from .config import (
    DATA_DIR,
    ID_COL,
    KAGGLE_INPUT,
    LABELS,
    PROMPT_COL,
    TARGET_COL,
    TEXT_COL,
    competition_dirs,
    resolve_path,
)
from .config import mount_summary as config_mount_summary


def as_text_array(series: pd.Series) -> np.ndarray:
    """Return a plain numpy object array of strings.

    pandas 3.x defaults string columns to a pyarrow-backed dtype whose
    ``.values`` is not fancy-indexable by sklearn's parallel CV workers and
    does not support 2-D numpy broadcasting.
    """
    return series.fillna("").astype(object).to_numpy(dtype=object)


def load_train() -> pd.DataFrame:
    """Load train.csv (2,699 passages / 150 prompts) with dtypes fixed."""
    df = pd.read_csv(
        resolve_path("train.csv"),
        dtype={ID_COL: str, PROMPT_COL: str, TEXT_COL: str, TARGET_COL: str},
    )
    return df


def load_test() -> pd.DataFrame:
    """Load test.csv (3 example rows locally / ~4,000 hidden rows on Kaggle)."""
    df = pd.read_csv(
        resolve_path("test.csv"),
        dtype={ID_COL: str, TEXT_COL: str},
    )
    return df


def load_sample_submission() -> pd.DataFrame:
    """Load sample_submission.parquet."""
    return pd.read_parquet(resolve_path("sample_submission.parquet"))


# ---------------------------------------------------------------------------
# Public "in-style" passage pools
# ---------------------------------------------------------------------------
#
# The competition's train set is only 2,699 passages.  Public datasets exist
# that hold tens of thousands of passages *generated by the same models under
# the same protocol* (e.g. ``prvsiyan/ntl-public-in-style-pool-v1`` on
# Kaggle).  Attaching one of them is the single largest known lever, because
# everything in this package — views, grouping, centring, assignment —
# transfers unchanged to a larger training set; only the loader is new.

IN_STYLE_TEXT_COLUMNS = ("text", "passage", "response", "output", "completion", "answer", "content")
IN_STYLE_MODEL_COLUMNS = ("model", "model_name", "author", "source", "llm", "generator")
IN_STYLE_PROMPT_COLUMNS = ("prompt_id", "prompt", "prompt_text", "question", "instruction")


def kaggle_input_dirs(root: Path | None = None) -> list[Path]:
    """List the *dataset roots* mounted under ``/kaggle/input``.

    Kaggle has used two layouts, and both are in the wild:

    * legacy:  ``/kaggle/input/<slug>/``
    * current: ``/kaggle/input/datasets/<owner>/<slug>/``

    The competition's own data is not returned by the legacy layout
    (``/kaggle/input/competitions/`` is skipped); under the current layout
    the competition appears as just another dataset and *is* returned, so
    filter with :func:`ntl.config.competition_dirs` when that matters.
    """
    root = Path(root) if root is not None else KAGGLE_INPUT
    if not root.exists():
        return []
    found: list[Path] = []
    for child in sorted(root.iterdir()):
        if not child.is_dir():
            continue
        if child.name == "datasets":
            # current layout: /kaggle/input/datasets/<owner>/<slug>/
            for owner in sorted(child.iterdir()):
                if owner.is_dir():
                    found.extend(sorted(p for p in owner.iterdir() if p.is_dir()))
        elif child.name != "competitions":
            found.append(child)
    return found


def _first_present(columns: list[str], candidates: tuple[str, ...]) -> str | None:
    lowered = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand in lowered:
            return lowered[cand]
    return None


def _read_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() in (".parquet", ".pq"):
        return pd.read_parquet(path)
    if path.suffix.lower() == ".jsonl":
        return pd.read_json(path, lines=True)
    return pd.read_csv(path)


def load_in_style_pool(
    roots: Sequence[Path] | None = None,
    pattern: str | None = None,
    max_files: int | None = None,
    exclude_competition: bool = True,
) -> pd.DataFrame:
    """Load and normalise a public in-style passage pool.

    Scans the given directories for parquet/CSV/JSONL files, normalises each
    table onto the competition's own schema (``id``, ``prompt_id``, ``text``,
    ``model``) and concatenates them.

    Column names are matched case-insensitively against a small list of
    plausible names, because public pools vary.  A file whose text column
    cannot be identified is skipped.

    Parameters
    ----------
    roots:
        Directories to search.  Defaults to every mounted dataset root
        (see :func:`kaggle_input_dirs`).
    pattern:
        Optional case-insensitive substring filter on the dataset folder
        name, e.g. ``"name-that-llm"``.  ``None`` means "every dataset".
    max_files:
        Stop after this many files (useful for a quick probe).
    exclude_competition:
        Skip the competition's own directory, so the competition's
        ``train.csv`` is never mistaken for pool data.

    Returns
    -------
    A frame with columns ``id``, ``prompt_id``, ``text``, ``model``.
    """
    if roots is None:
        roots = kaggle_input_dirs()

    competition = set(competition_dirs()[:1])
    selected: list[Path] = []
    for root in roots:
        if exclude_competition and root in competition:
            continue
        if pattern and pattern.lower() not in str(root).lower():
            continue
        selected.append(root)
    frames: list[pd.DataFrame] = []
    files = sorted(
        p
        for root in selected
        for p in root.rglob("*")
        if p.suffix.lower() in (".parquet", ".pq", ".csv", ".jsonl")
        and not any(c in p.parents for c in competition)
    )
    if max_files is not None:
        files = files[:max_files]

    for path in files:
        try:
            raw = _read_table(path)
        except Exception:  # noqa: BLE001 - a malformed shard must not kill the load
            continue
        text_col = _first_present(list(raw.columns), IN_STYLE_TEXT_COLUMNS)
        if text_col is None:
            continue
        model_col = _first_present(list(raw.columns), IN_STYLE_MODEL_COLUMNS)
        prompt_col = _first_present(list(raw.columns), IN_STYLE_PROMPT_COLUMNS)
        frame = pd.DataFrame(
            {
                "text": raw[text_col].astype(str),
                "model": raw[model_col].astype(str) if model_col else "",
                "prompt_id": raw[prompt_col].astype(str) if prompt_col else "",
            }
        )
        frames.append(frame)

    if not frames:
        return pd.DataFrame(columns=["id", "prompt_id", "text", "model"])

    pool = pd.concat(frames, ignore_index=True)
    pool = pool[pool["text"].str.len() > 0].reset_index(drop=True)
    pool.insert(0, "id", [f"pool-{i:06d}" for i in range(len(pool))])
    return pool[["id", "prompt_id", "text", "model"]]


def describe_in_style_pool(pool: pd.DataFrame) -> dict:
    """Summarise a loaded pool: size, per-model counts, and label coverage.

    ``label_coverage`` is the share of pool rows whose ``model`` label is one
    of the 18 competition labels — a quick check that the pool speaks the
    same vocabulary as the task before spending a training run on it.
    """
    if pool.empty:
        return {"rows": 0, "models": {}, "label_coverage": 0.0}
    counts = pool["model"].value_counts()
    known = sum(int(counts.get(m, 0)) for m in LABELS)
    return {
        "rows": int(len(pool)),
        "models": {str(k): int(v) for k, v in counts.items()},
        "label_coverage": known / len(pool),
    }


def prepare_training_pool(
    pool: pd.DataFrame,
    labels: Sequence[str] | None = None,
    train_texts: Sequence[str] | None = None,
    prompt_prefix: str = "pool:",
) -> pd.DataFrame:
    """Make a public in-style pool safe to concatenate with the train set.

    A raw pool is *not* drop-in training data.  Four things go wrong if it is
    used as-is, and all four are silent:

    1. **Unknown labels.**  Public pools often carry extra models (``gpt-4``,
       ``claude-2``, ...).  Training on them adds classes the competition never
       scores and makes the submission invalid.
    2. **Duplicate passages.**  A pool may re-publish the competition's own
       train rows.  Training on a copy would inflate CV and leak.
    3. **Colliding prompt ids.**  ``prompt_007`` in the pool is almost
       certainly not ``prompt_007`` in train; merging them would centre two
       different prompts together.
    4. **Missing prompt ids.**  Some pools ship text + model only.  Giving
       every such row its *own* group keeps it in the classifier while
       excluding it from group centring (groups of one are not centred).

    Returns a frame with the usual ``id``, ``prompt_id``, ``text``, ``model``
    columns, deduplicated and label-filtered.  Safe on an empty frame.
    """
    empty = pd.DataFrame(columns=["id", "prompt_id", "text", "model"])
    if pool is None or pool.empty:
        return empty

    known = tuple(labels) if labels is not None else tuple(LABELS)
    frame = pool.copy()
    frame = frame[frame["model"].astype(str).isin(known)]
    frame = frame[frame["text"].astype(str).str.len() > 0]

    if train_texts is not None:
        seen = set(map(str, train_texts))
        frame = frame[~frame["text"].astype(str).isin(seen)]

    frame = frame.reset_index(drop=True)
    prompt = frame["prompt_id"].fillna("").astype(str)
    namespaced = [
        f"{prompt_prefix}{value}" if value else f"{prompt_prefix}row:{i}"
        for i, value in enumerate(prompt)
    ]
    frame["prompt_id"] = namespaced
    frame["id"] = frame["id"].astype(str)
    return frame[["id", "prompt_id", "text", "model"]].reset_index(drop=True)


def mount_summary(max_depth: int = 3) -> list[str]:
    """A readable dump of the Kaggle mount.

    Delegates to :func:`ntl.config.mount_summary`; kept here so notebooks can
    call ``data.mount_summary()`` without importing ``ntl.config`` directly.
    """
    return config_mount_summary(max_depth=max_depth)


def data_locations() -> dict[str, list[str]]:
    """Where the loader thinks the data lives.

    Returns the competition directories that were discovered, the attached
    dataset roots, and the local data directory — as plain strings so the
    result is easy to print in a notebook.
    """
    return {
        "competition_dirs": [str(p) for p in competition_dirs()],
        "attached_datasets": [str(p) for p in kaggle_input_dirs()],
        "local_data_dir": str(DATA_DIR),
    }


def inventory() -> dict[str, dict]:
    """Report what data files exist and their shapes/columns.

    Returns a dict keyed by file name with ``exists``, ``rows``, ``columns``.
    """
    report: dict[str, dict] = {}
    for name in ("train.csv", "test.csv", "sample_submission.parquet"):
        entry: dict = {"exists": False, "rows": None, "columns": None, "path": None}
        for base in [DATA_DIR, *competition_dirs()]:
            path = base / name
            if path.exists():
                entry.update(exists=True, path=str(path))
                if name.endswith(".csv"):
                    df = pd.read_csv(path, nrows=5)
                else:
                    df = pd.read_parquet(path)
                entry["rows"] = len(pd.read_csv(path) if name.endswith(".csv") else df)
                entry["columns"] = list(df.columns)
                break
        report[name] = entry
    return report
# %% module: features
"""Lightweight stylistic features for LLM attribution.

The competition blurb hints the signal lives in *style*: word choice,
punctuation, sentence structure — not topic or length shortcuts. These are
cheap, model-agnostic features we can explore (and later feed to classical
classifiers) before reaching for embeddings.
"""

from __future__ import annotations

import re

import pandas as pd

_WORD_RE = re.compile(r"[A-Za-z']+")
_SENT_RE = re.compile(r"[.!?]+")

# Punctuation probes: each lab's post-training tends to leave fingerprints here.
_PUNCT = {
    "comma": ",",
    "period": ".",
    "exclamation": "!",
    "question": "?",
    "semicolon": ";",
    "colon": ":",
    "dash": "—",  # em dash
    "hyphen": "-",
    "paren": "()",
    "quote": "\"'",
    "ellipsis": "...",
    "asterisk": "*",
    "hash": "#",
}


def add_text_features(df: pd.DataFrame, text_col: str = "text") -> pd.DataFrame:
    """Return a copy of *df* with stylistic feature columns appended.

    Features cover length, lexical richness, sentence shape, punctuation
    habits and markdown artifacts (headers, bullets, bold).
    """
    out = df.copy()
    texts = out[text_col].fillna("")

    out["n_chars"] = texts.str.len()
    out["n_letters"] = texts.str.count(r"[A-Za-z]")
    out["n_digits"] = texts.str.count(r"\d")
    out["n_upper"] = texts.str.count(r"[A-Z]")
    out["n_whitespace"] = texts.str.count(r"\s")
    out["n_newline"] = texts.str.count(r"\n")

    words = texts.str.findall(_WORD_RE)
    out["n_words"] = words.apply(len)
    out["avg_word_len"] = words.apply(lambda ws: sum(len(w) for w in ws) / len(ws) if ws else 0.0)
    out["lexical_diversity"] = words.apply(
        lambda ws: len(set(w.lower() for w in ws)) / len(ws) if ws else 0.0
    )

    sents = texts.str.findall(_SENT_RE)
    out["n_sentences"] = sents.apply(len)
    out["avg_sent_len"] = out["n_words"] / out["n_sentences"].clip(lower=1)

    for name, char in _PUNCT.items():
        out[f"punct_{name}"] = texts.str.count(re.escape(char)) / out["n_words"].clip(lower=1)

    # Markdown / structure artifacts (only "light cleanup" was applied per the
    # data description, so in-passage markdown likely survives).
    out["md_header"] = texts.str.count(r"(?m)^#{1,6}\s")
    out["md_bullet"] = texts.str.count(r"(?m)^\s*([-*•]|\d+\.)\s")
    out["md_bold"] = texts.str.count(r"\*\*|__")
    out["md_code"] = texts.str.count(r"`")
    out["md_math"] = texts.str.count(r"\\\[|\\\(|\$")

    return out


FEATURE_COLS: list[str] = [
    "n_chars",
    "n_letters",
    "n_digits",
    "n_upper",
    "n_newline",
    "n_words",
    "avg_word_len",
    "lexical_diversity",
    "n_sentences",
    "avg_sent_len",
    *[f"punct_{name}" for name in _PUNCT],
    "md_header",
    "md_bullet",
    "md_bold",
    "md_code",
    "md_math",
]
# %% module: views
"""Linguistic *views*: different reductions of a passage into a feature space.

Why views at all?  Every passage answers a prompt, so a large part of its
surface form is chosen by the *prompt*, not by the *model*.  Vocabulary is
therefore the worst possible space to identify an author in: two models
answering the same prompt will agree on the nouns.  What differs between
sibling models of the same lab is *how* the passage is put together.

Each view below removes a different layer of topical content and leaves a
different layer of style:

============  =========================================================
view          what survives
============  =========================================================
``char``      everything: characters, words, punctuation (lexical + format)
``word``      word n-grams: vocabulary and word order
``shape``     word *shape* only (``X``/``x``): capitalisation habits
``funcon``    syntax only: function words kept, content words masked
``punct``     formatting only: punctuation and whitespace
============  =========================================================

``funcon`` and ``shape`` are the interesting ones, because they are
*invariant to topic*: two passages answering different prompts with the same
rhetorical structure look the same under them.  That is exactly what we
want when the prompt is a nuisance variable to be removed.
"""

from __future__ import annotations

import re
from collections.abc import Callable, Iterable, Sequence
from dataclasses import dataclass

import numpy as np
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

# Closed-class words.  These carry grammar (who did what to whom, how certain,
# how connected), not topic.  A list rather than a library stop-word list, so
# that *content* words like "nothing" or "never" stay visible where they are
# meaningful as hedges.
FUNCTION_WORDS: frozenset[str] = frozenset(
    [
        "a",
        "an",
        "the",
        "and",
        "or",
        "but",
        "if",
        "while",
        "although",
        "because",
        "as",
        "so",
        "than",
        "that",
        "which",
        "who",
        "whom",
        "whose",
        "this",
        "these",
        "those",
        "it",
        "its",
        "they",
        "them",
        "their",
        "theirs",
        "he",
        "him",
        "his",
        "she",
        "her",
        "hers",
        "we",
        "us",
        "our",
        "ours",
        "you",
        "your",
        "yours",
        "i",
        "me",
        "my",
        "mine",
        "in",
        "on",
        "at",
        "to",
        "from",
        "with",
        "without",
        "for",
        "of",
        "by",
        "over",
        "under",
        "between",
        "through",
        "into",
        "about",
        "against",
        "during",
        "before",
        "after",
        "is",
        "are",
        "was",
        "were",
        "be",
        "been",
        "being",
        "am",
        "have",
        "has",
        "had",
        "having",
        "do",
        "does",
        "did",
        "doing",
        "done",
        "can",
        "could",
        "will",
        "would",
        "shall",
        "should",
        "may",
        "might",
        "must",
        "ought",
        "need",
        "dare",
        "not",
        "no",
        "nor",
        "none",
        "never",
        "neither",
        "none",
        "nothing",
        "nowhere",
        "anyone",
        "anything",
        "everyone",
        "everything",
        "all",
        "any",
        "some",
        "each",
        "every",
        "both",
        "either",
        "neither",
        "other",
        "another",
        "such",
        "same",
        "own",
        "few",
        "little",
        "several",
        "much",
        "many",
        "more",
        "most",
        "less",
        "least",
        "also",
        "just",
        "only",
        "even",
        "still",
        "yet",
        "already",
        "always",
        "never",
        "often",
        "sometimes",
        "usually",
        "rarely",
        "very",
        "too",
        "here",
        "there",
        "when",
        "where",
        "why",
        "how",
        "what",
        "whether",
        "which",
        "because",
        "therefore",
        "thus",
        "hence",
        "consequently",
        "moreover",
        "furthermore",
        "however",
        "nevertheless",
        "nonetheless",
        "here",
        "now",
        "then",
        "since",
        "until",
        "while",
        "during",
        "before",
        "after",
        "above",
        "below",
        "between",
        "within",
        "without",
        "upon",
        "toward",
        "towards",
        "across",
        "around",
        "of",
        "off",
        "out",
        "up",
        "down",
        "in",
        "on",
        "at",
    ]
)

_TOKEN_RE = re.compile(r"\w+|[^\w\s]")
_WORD_RE = re.compile(r"[^\W\d_]+")


def shape_transform(text: str) -> str:
    """Replace every alphabetic word by ``X`` (if capitalised) or ``x``.

    Keeps capitalisation, word length and punctuation; throws away identity.
    """
    return _WORD_RE.sub(lambda m: "X" if m.group(0)[0].isupper() else "x", text)


def function_transform(text: str) -> str:
    """Keep function words and punctuation, mask every content word.

    The result is a *syntactic skeleton* of the passage: the same rhetorical
    move ("however, ... therefore, ... it is ...") survives regardless of what
    the passage is about.
    """
    out: list[str] = []
    for tok in _TOKEN_RE.findall(text):
        if not tok.isalnum():
            out.append(tok)
            continue
        low = tok.lower()
        out.append(low if low in FUNCTION_WORDS else "CONTENT")
    return " ".join(out)


def punct_transform(text: str) -> str:
    """Strip letters and digits; keep punctuation and whitespace only.

    This is the purest formatting view: dashes, commas, quotes, line breaks.
    """
    return re.sub(r"[A-Za-z0-9]+", " ", text)


@dataclass(frozen=True)
class ViewSpec:
    """A named text transform plus the TF-IDF settings used on top of it."""

    name: str
    transform: Callable[[str], str]
    vectorizer_kwargs: dict

    def apply(self, text: str) -> str:
        return self.transform(text)


VIEWS: dict[str, ViewSpec] = {
    "char": ViewSpec(
        "char",
        lambda t: t,
        dict(
            analyzer="char_wb",
            ngram_range=(2, 5),
            min_df=3,
            sublinear_tf=True,
            lowercase=False,
            max_features=300_000,
        ),
    ),
    "word": ViewSpec(
        "word",
        lambda t: t,
        dict(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
            lowercase=True,
            max_features=200_000,
            token_pattern=r"(?u)\b\w+\b|[^\w\s]",
        ),
    ),
    "shape": ViewSpec(
        "shape",
        shape_transform,
        dict(
            analyzer="char",
            ngram_range=(2, 6),
            min_df=3,
            sublinear_tf=True,
            lowercase=False,
            max_features=100_000,
        ),
    ),
    "funcon": ViewSpec(
        "funcon",
        function_transform,
        dict(
            analyzer="word",
            ngram_range=(1, 3),
            min_df=3,
            sublinear_tf=True,
            lowercase=True,
            max_features=100_000,
            token_pattern=r"(?u)\b\w+\b|[^\w\s]",
        ),
    ),
    "punct": ViewSpec(
        "punct",
        punct_transform,
        dict(
            analyzer="char",
            ngram_range=(1, 4),
            min_df=3,
            sublinear_tf=True,
            lowercase=False,
            max_features=50_000,
        ),
    ),
}

DEFAULT_VIEWS: tuple[str, ...] = ("char", "word", "funcon")


def view_text(view: str, text: str) -> str:
    """Apply the transform of a named view to one text."""
    return VIEWS[view].apply(text)


def fit_views(
    texts: Sequence[str], views: Iterable[str] = DEFAULT_VIEWS
) -> dict[str, TfidfVectorizer]:
    """Fit one TF-IDF vectorizer per view on *texts*."""
    fitted: dict[str, TfidfVectorizer] = {}
    for name in views:
        spec = VIEWS[name]
        vec = TfidfVectorizer(dtype=np.float32, **spec.vectorizer_kwargs)
        vec.fit([spec.apply(t) for t in texts])
        fitted[name] = vec
    return fitted


def transform_views(
    fitted: dict[str, TfidfVectorizer],
    texts: Sequence[str],
    views: Iterable[str] = DEFAULT_VIEWS,
    normalize_rows: bool = True,
) -> sparse.csr_matrix:
    """Vectorise *texts* under every view and stack the blocks.

    Each block is L2-normalised so that no view dominates the shared weight
    vector purely because it has more features.
    """
    blocks = []
    for name in views:
        spec = VIEWS[name]
        block = fitted[name].transform([spec.apply(t) for t in texts])
        blocks.append(normalize(block) if normalize_rows else block)
    return sparse.hstack(blocks, format="csr")


class ViewUnion(BaseEstimator, TransformerMixin):
    """scikit-learn transformer exposing several views as one feature space.

    Lets a view stack be dropped straight into a ``Pipeline``.
    """

    def __init__(self, views: Sequence[str] | None = None):
        self.views = list(views) if views is not None else list(DEFAULT_VIEWS)

    def fit(self, X, y=None):  # noqa: N803 - sklearn convention
        self.vecs_ = fit_views(X, self.views)
        return self

    def transform(self, X):  # noqa: N803 - sklearn convention
        return transform_views(self.vecs_, X, self.views)

    def fit_transform(self, X, y=None, **fit_params):  # noqa: N803
        self.vecs_ = fit_views(X, self.views)
        return transform_views(self.vecs_, X, self.views)

    def get_feature_names_out(self, input_features=None):
        names: list[str] = []
        for name in self.views:
            names.extend(f"{name}__{f}" for f in self.vecs_[name].get_feature_names_out())
        return np.array(names, dtype=object)
# %% module: facets
"""Facets — the checklist of style questions, turned into numbers.

The competition gives the response text and nothing else: ``test.csv`` has
``id, text``.  So a facet is *usable* only if it can be computed from the
response alone, or from the response plus the prompt groups this package
recovers transductively.  That one constraint decides the whole design:

======================  ===========================  ==========================
question                computable at test time?      how
======================  ===========================  ==========================
words and phrases       yes                          n-grams (:mod:`ntl.views`)
punctuation             yes                          per-100-word rates
sentence length         yes                          sentence split
repeated words          yes                          type-token ratio, hapax
structure (bullets ...) yes                          markdown probes
emojis                  yes                          unicode ranges
wording (person)        yes                          pronoun rates
hedging / boosters      yes                          closed word lists
citations               yes                          bracket / year probes
response length         yes                          counts
deviation from prompt   *proxy*                      cosine to the recovered
                                                      group centroid
prompt domain           *proxy*                      centroid topic — but it is
                                                      constant inside a group
prompt clarity / shots  **no**                       the prompt is not in test
prompt-response distance**no**                       no prompt text at test
======================  ===========================  ==========================

Two consequences follow, and they explain the two halves of this module:

* :func:`response_facets` is the honest core: every column is a function of
  the passage alone, so it transfers to the hidden test unchanged.
* :func:`group_facets` is the transductive bonus: it compares a passage with
  the *recovered* prompt group it belongs to.  It cannot separate passages
  whose groups are already merged, but it does capture "this answer is
  unusually long for this question", which raw rates cannot.

Prompt-side facets (domain, clarity, few-shot, guardrails) are deliberately
absent because they are not observable at prediction time.  They can still
shape *training* (domain-stratified models), but :mod:`ntl.measure` shows the
inferred domain is constant within a recovered group, so it carries no
within-group signal and cannot identify an author.

Naming: ``f_`` columns are response facets, ``g_`` columns are group facets.
"""

from __future__ import annotations

import re
from collections.abc import Iterable, Sequence

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.preprocessing import normalize

# ---------------------------------------------------------------------------
# Lexical resources
# ---------------------------------------------------------------------------

_WORD_RE = re.compile(r"[A-Za-z][A-Za-z'-]*")
_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")
_EMOJI_RE = re.compile(
    r"[\U0001f300-\U0001faff\U00002600-\U000027bf\U0001f1e6-\U0001f1ff"
    r"\U00002190-\U000021ff\U00002b00-\U00002bff]"
)

FIRST_SINGULAR = frozenset({"i", "me", "my", "mine", "myself"})
FIRST_PLURAL = frozenset({"we", "us", "our", "ours", "ourselves"})
SECOND_PERSON = frozenset({"you", "your", "yours", "yourself", "yourselves"})
THIRD_PERSON = frozenset(
    {"he", "him", "his", "she", "her", "hers", "they", "them", "their", "theirs"}
)

HEDGES = frozenset(
    {
        "perhaps",
        "maybe",
        "possibly",
        "probably",
        "likely",
        "arguably",
        "seemingly",
        "apparently",
        "roughly",
        "approximately",
        "somewhat",
        "fairly",
        "relatively",
        "generally",
        "typically",
        "often",
        "sometimes",
        "tends",
        "seems",
        "appears",
        "suggests",
        "may",
        "might",
        "could",
    }
)
BOOSTERS = frozenset(
    {
        "clearly",
        "certainly",
        "definitely",
        "indeed",
        "obviously",
        "undoubtedly",
        "absolutely",
        "always",
        "never",
        "must",
        "crucially",
        "importantly",
        "notably",
        "especially",
        "particularly",
    }
)
CONNECTIVES = frozenset(
    {
        "however",
        "therefore",
        "moreover",
        "furthermore",
        "nevertheless",
        "nonetheless",
        "consequently",
        "thus",
        "hence",
        "additionally",
        "meanwhile",
        "similarly",
        "conversely",
        "specifically",
        "ultimately",
        "overall",
        "finally",
        "first",
        "second",
        "third",
    }
)

_CONTRACTION_RE = re.compile(r"\b(?:[A-Za-z]+)'(?:s|t|re|ve|ll|d|m)\b", re.IGNORECASE)
_CITE_BRACKET_RE = re.compile(r"\[\d{1,3}\]")
_CITE_YEAR_RE = re.compile(r"\((?:[A-Z][A-Za-z.\-]+(?: et al\.)?,\s*)?\d{4}[a-z]?\)")
_URL_RE = re.compile(r"https?://\S+")
_ALLCAPS_RE = re.compile(r"\b[A-Z]{2,}\b")

_MD_HEADER_RE = re.compile(r"(?m)^#{1,6}\s")
_MD_BULLET_RE = re.compile(r"(?m)^\s*[-*\u2022]\s")
_MD_NUMBERED_RE = re.compile(r"(?m)^\s*\d+[.)]\s")
_MD_TABLE_RE = re.compile(r"(?m)^\s*\|.*\|\s*$")
_MD_BOLD_RE = re.compile(r"\*\*|__")
_MD_CODE_RE = re.compile(r"```|`[^`\n]+`")
_MD_LINK_RE = re.compile(r"\[[^\]]+\]\([^)]+\)")

_PUNCTUATION: dict[str, str] = {
    "comma": ",",
    "semicolon": ";",
    "colon": ":",
    "emdash": "\u2014",
    "endash": "\u2013",
    "paren": "()",
    "quote": "\"'",
    "exclam": "!",
    "question": "?",
    "ellipsis": "...",
    "slash": "/",
    "percent": "%",
}

# ---------------------------------------------------------------------------
# Feature families — the vocabulary the notebooks and the tracker speak
# ---------------------------------------------------------------------------

LENGTH_FACETS = (
    "f_words",
    "f_chars",
    "f_sentences",
    "f_paragraphs",
    "f_words_per_sentence",
    "f_sentence_len_sd",
    "f_avg_word_len",
    "f_long_word_rate",
    "f_short_word_rate",
)
LEXICAL_FACETS = ("f_ttr", "f_hapax_rate", "f_stopword_rate", "f_contraction_rate")
PUNCTUATION_FACETS = tuple(f"f_punct_{name}" for name in _PUNCTUATION)
STRUCTURE_FACETS = (
    "f_md_header",
    "f_md_bullet",
    "f_md_numbered",
    "f_md_table",
    "f_md_code",
    "f_md_bold",
    "f_md_link",
    "f_newline_rate",
    "f_digits_rate",
    "f_allcaps_rate",
)
EMOJI_FACETS = ("f_emoji_rate",)
PERSON_FACETS = (
    "f_first_singular",
    "f_first_plural",
    "f_second_person",
    "f_third_person",
    "f_impersonal_one",
)
HEDGING_FACETS = ("f_hedge_rate", "f_booster_rate", "f_connective_rate")
CITATION_FACETS = ("f_cite_bracket", "f_cite_year", "f_url")
OPENER_FACETS = (
    "f_open_it",
    "f_open_there",
    "f_open_the",
    "f_open_wh",
    "f_question_sentence_rate",
)
GROUP_FACETS = (
    "g_word_rank",
    "g_word_z",
    "g_sentence_z",
    "g_ttr_z",
    "g_comma_z",
    "g_len_ratio",
    "g_centroid_cos",
    "g_size_log",
)

FAMILIES: dict[str, tuple[str, ...]] = {
    "length": LENGTH_FACETS,
    "lexical": LEXICAL_FACETS,
    "punctuation": PUNCTUATION_FACETS,
    "structure": STRUCTURE_FACETS,
    "emoji": EMOJI_FACETS,
    "person": PERSON_FACETS,
    "hedging": HEDGING_FACETS,
    "citation": CITATION_FACETS,
    "opener": OPENER_FACETS,
    "group": GROUP_FACETS,
}

RESPONSE_FAMILIES: tuple[str, ...] = tuple(f for f in FAMILIES if f != "group")
ALL_RESPONSE_FACETS: tuple[str, ...] = tuple(
    column for family in RESPONSE_FAMILIES for column in FAMILIES[family]
)
ALL_FACETS: tuple[str, ...] = tuple(dict.fromkeys(ALL_RESPONSE_FACETS + GROUP_FACETS))

_FAMILY_OF: dict[str, str] = {
    column: family for family, cols in FAMILIES.items() for column in cols
}


def family_of(column: str) -> str:
    """Which family a facet column belongs to (``"unknown"`` if unregistered)."""
    return _FAMILY_OF.get(column, "unknown")


def facet_columns(families: Iterable[str] | None = None) -> list[str]:
    """Resolve family names *and/or* column names into an ordered column list.

    ``facet_columns(["structure", "g_centroid_cos"])`` is valid and returns
    the structure family followed by that one group facet.  Unknown names
    raise, so a typo in a notebook fails loudly instead of silently training
    on nothing.
    """
    if families is None:
        return list(ALL_FACETS)
    out: list[str] = []
    for name in families:
        if name in FAMILIES:
            out.extend(FAMILIES[name])
        elif name in ALL_FACETS:
            out.append(name)
        else:
            raise KeyError(f"unknown facet family/column {name!r}")
    return list(dict.fromkeys(out))


# ---------------------------------------------------------------------------
# Response facets
# ---------------------------------------------------------------------------


def _rate(count: float, n_words: int) -> float:
    return 100.0 * float(count) / max(n_words, 1)


def _analyse(text: str) -> dict[str, float]:
    """Every response-side facet for one passage."""
    text = "" if text is None else str(text)
    words = _WORD_RE.findall(text)
    lower = [w.lower() for w in words]
    n_words = len(words)
    n_chars = len(text)

    sentences = [s for s in _SENT_SPLIT.split(text.strip()) if s.strip()]
    n_sentences = len(sentences)
    sent_lens = [len(_WORD_RE.findall(s)) for s in sentences] or [0]
    paragraphs = [p for p in re.split(r"\n\s*\n", text.strip()) if p.strip()]

    first_100 = lower[:100]
    ttr = len(set(first_100)) / len(first_100) if first_100 else 0.0
    counts: dict[str, int] = {}
    for word in lower:
        counts[word] = counts.get(word, 0) + 1
    hapax = sum(1 for c in counts.values() if c == 1)
    stop = sum(1 for w in lower if w in ENGLISH_STOP_WORDS)

    openers = [_WORD_RE.findall(s)[:1] for s in sentences]
    openers = [o[0].lower() for o in openers if o]
    n_openers = max(len(openers), 1)

    row: dict[str, float] = {
        "f_words": float(n_words),
        "f_chars": float(n_chars),
        "f_sentences": float(n_sentences),
        "f_paragraphs": float(len(paragraphs)),
        "f_words_per_sentence": n_words / max(n_sentences, 1),
        "f_sentence_len_sd": float(np.std(sent_lens)) if len(sent_lens) > 1 else 0.0,
        "f_avg_word_len": float(np.mean([len(w) for w in words])) if words else 0.0,
        "f_long_word_rate": _rate(sum(1 for w in words if len(w) >= 8), n_words),
        "f_short_word_rate": _rate(sum(1 for w in words if len(w) <= 3), n_words),
        "f_ttr": ttr,
        "f_hapax_rate": hapax / max(n_words, 1),
        "f_stopword_rate": stop / max(n_words, 1),
        "f_contraction_rate": _rate(len(_CONTRACTION_RE.findall(text)), n_words),
        "f_md_header": float(len(_MD_HEADER_RE.findall(text))),
        "f_md_bullet": float(len(_MD_BULLET_RE.findall(text))),
        "f_md_numbered": float(len(_MD_NUMBERED_RE.findall(text))),
        "f_md_table": float(len(_MD_TABLE_RE.findall(text))),
        "f_md_code": float(len(_MD_CODE_RE.findall(text))),
        "f_md_bold": float(len(_MD_BOLD_RE.findall(text))),
        "f_md_link": float(len(_MD_LINK_RE.findall(text))),
        "f_newline_rate": _rate(text.count("\n"), n_words),
        "f_digits_rate": _rate(sum(ch.isdigit() for ch in text), n_words),
        "f_allcaps_rate": _rate(len(_ALLCAPS_RE.findall(text)), n_words),
        "f_emoji_rate": _rate(len(_EMOJI_RE.findall(text)), n_words),
        "f_first_singular": _rate(sum(w in FIRST_SINGULAR for w in lower), n_words),
        "f_first_plural": _rate(sum(w in FIRST_PLURAL for w in lower), n_words),
        "f_second_person": _rate(sum(w in SECOND_PERSON for w in lower), n_words),
        "f_third_person": _rate(sum(w in THIRD_PERSON for w in lower), n_words),
        "f_impersonal_one": _rate(sum(w == "one" for w in lower), n_words),
        "f_hedge_rate": _rate(sum(w in HEDGES for w in lower), n_words),
        "f_booster_rate": _rate(sum(w in BOOSTERS for w in lower), n_words),
        "f_connective_rate": _rate(sum(w in CONNECTIVES for w in lower), n_words),
        "f_cite_bracket": _rate(len(_CITE_BRACKET_RE.findall(text)), n_words),
        "f_cite_year": _rate(len(_CITE_YEAR_RE.findall(text)), n_words),
        "f_url": _rate(len(_URL_RE.findall(text)), n_words),
        "f_open_it": sum(o == "it" for o in openers) / n_openers,
        "f_open_there": sum(o == "there" for o in openers) / n_openers,
        "f_open_the": sum(o == "the" for o in openers) / n_openers,
        "f_open_wh": sum(o in {"what", "why", "how", "when", "where", "who"} for o in openers)
        / n_openers,
        "f_question_sentence_rate": sum(s.rstrip().endswith("?") for s in sentences)
        / max(n_sentences, 1),
    }
    for name, chars in _PUNCTUATION.items():
        row[f"f_punct_{name}"] = _rate(sum(text.count(ch) for ch in chars), n_words)
    return row


def response_facets(texts: Sequence[str]) -> pd.DataFrame:
    """One row of response-only facets per passage.

    Every column is computable from the hidden test set, because the hidden
    test set contains exactly this input: the passage text.
    """
    rows = [_analyse(text) for text in texts]
    frame = pd.DataFrame(rows)
    return frame.reindex(columns=list(ALL_RESPONSE_FACETS), fill_value=0.0).astype(float)


# ---------------------------------------------------------------------------
# Group facets (transductive)
# ---------------------------------------------------------------------------


def _group_rank(values: np.ndarray, groups: np.ndarray) -> np.ndarray:
    out = np.full(len(values), 0.5)
    for group in np.unique(groups):
        idx = np.flatnonzero(groups == group)
        if len(idx) < 2:
            continue
        order = np.argsort(values[idx], kind="stable")
        ranks = np.empty(len(idx))
        ranks[order] = np.arange(len(idx)) / (len(idx) - 1)
        out[idx] = ranks
    return out


def _group_z(values: np.ndarray, groups: np.ndarray) -> np.ndarray:
    out = np.zeros(len(values))
    for group in np.unique(groups):
        idx = np.flatnonzero(groups == group)
        if len(idx) < 2:
            continue
        block = values[idx]
        sd = float(block.std())
        out[idx] = 0.0 if sd < 1e-12 else (block - block.mean()) / sd
    return out


def _centroid_similarity(texts: Sequence[str], groups: np.ndarray) -> np.ndarray:
    """Leave-one-out cosine between a passage and its group's content centroid.

    This is the computable stand-in for "distance from the prompt": the group
    centroid is what all answers to one prompt share, i.e. a reconstruction of
    the prompt itself.  A passage close to the centroid answered the question
    on topic; a far one drifted.  Self is removed so that a two-member group
    cannot look artificially typical.
    """
    n = len(texts)
    if n < 2:
        return np.ones(n)
    vec = TfidfVectorizer(stop_words="english", sublinear_tf=True, dtype=np.float32)
    X = normalize(vec.fit_transform(texts))
    _, inverse = np.unique(groups, return_inverse=True)
    membership = sparse.csr_matrix(
        (np.ones(n), (np.arange(n), inverse)), shape=(n, int(inverse.max()) + 1)
    )
    sums = membership.T @ X  # (n_groups, vocab)
    row_sums = sums[inverse]  # (n, vocab)
    dot_all = np.asarray(X.multiply(row_sums).sum(axis=1)).ravel()
    norm_sq = np.asarray(sums.multiply(sums).sum(axis=1)).ravel()[inverse]
    loo_dot = dot_all - 1.0  # x . (S - x) = x.S - 1
    loo_norm = np.sqrt(np.maximum(norm_sq - 2.0 * dot_all + 1.0, 0.0))
    cos = np.divide(loo_dot, loo_norm, out=np.zeros_like(loo_dot), where=loo_norm > 1e-9)
    return np.clip(cos, -1.0, 1.0)


def group_facets(
    texts: Sequence[str],
    groups: Sequence,
    base: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """Facets that describe a passage *relative to its recovered prompt group*.

    ``groups`` may be the true ``prompt_id`` (at training time) or the groups
    recovered by :func:`ntl.groups.infer_groups` (at prediction time).  The
    features are defined so that they mean the same thing either way.
    """
    groups = np.asarray(groups)
    base = response_facets(texts) if base is None else base
    words = base["f_words"].to_numpy(dtype=float)
    sentences = base["f_sentences"].to_numpy(dtype=float)
    ttr = base["f_ttr"].to_numpy(dtype=float)
    commas = base["f_punct_comma"].to_numpy(dtype=float)

    group_median = pd.Series(words).groupby(groups).transform("median").to_numpy(dtype=float)
    sizes = pd.Series(groups).map(pd.Series(groups).value_counts()).to_numpy(dtype=float)

    out = pd.DataFrame(index=base.index)
    out["g_word_rank"] = _group_rank(words, groups)
    out["g_word_z"] = _group_z(np.log1p(words), groups)
    out["g_sentence_z"] = _group_z(np.log1p(sentences), groups)
    out["g_ttr_z"] = _group_z(ttr, groups)
    out["g_comma_z"] = _group_z(commas, groups)
    out["g_len_ratio"] = words / np.maximum(group_median, 1.0)
    out["g_centroid_cos"] = _centroid_similarity(texts, groups)
    out["g_size_log"] = np.log1p(sizes)
    return out[list(GROUP_FACETS)].astype(float)


def facet_frame(
    texts: Sequence[str],
    groups: Sequence | None = None,
    families: Iterable[str] | None = None,
) -> pd.DataFrame:
    """The full facet frame for *texts*, restricted to *families*.

    With ``groups=None`` only response facets are returned; asking for the
    ``group`` family without groups is an error, because those columns would
    silently be meaningless.
    """
    columns = facet_columns(families)
    needs_group = any(c in GROUP_FACETS for c in columns)
    if needs_group and groups is None:
        raise ValueError("group facets require groups")
    base = response_facets(texts)
    if needs_group:
        base = pd.concat([base, group_facets(texts, groups, base)], axis=1)
    return base.reindex(columns=columns, fill_value=0.0).astype(float)
# %% module: stylometry
"""The verbal fingerprint: which words a model actually repeats.

Motivation
----------
If two models can be told apart at all, it should be by *habit*: the words
they reach for, the hedges they use, the way they open a sentence.  This
module measures that directly, with the standard statistical test for
"distinctive words" — the Monroe et al. (2008) log-odds ratio with an
informative Dirichlet prior.

Why not a plain frequency table?  Raw counts are dominated by passage length
and by the topic of the prompt.  The log-odds z-score asks a sharper
question: *given that this word appears at all, how much more often does
model A use it than the other seventeen?*  The Dirichlet prior keeps rare
words from producing infinite z-scores, which is what makes the ranking
trustworthy on a corpus this small.

Two things this analysis revealed (see the notebook for the evidence):

* Family-level fingerprints are strong and legible — Google models reach for
  "profound / absolute / void", OpenAI models for "therefore / seem / turn".
* Sibling models *within* a family are **not** separable by vocabulary: their
  top words differ only in topic, which is prompt noise, not style.

That asymmetry is the whole reason the pipeline needs topic-free views.
"""

from __future__ import annotations

import re
from collections import Counter
from collections.abc import Iterable, Sequence

import numpy as np
import pandas as pd

_TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z'-]+")

#: Closed-class words are excluded from the "vocabulary" analyses: they
#: belong to the *style* views (see :mod:`ntl.views`), not to the lexical one.
STOPWORDS: frozenset[str] = frozenset(
    [
        "a",
        "an",
        "the",
        "and",
        "or",
        "but",
        "if",
        "while",
        "although",
        "because",
        "as",
        "so",
        "than",
        "that",
        "which",
        "who",
        "whom",
        "whose",
        "this",
        "these",
        "those",
        "it",
        "its",
        "they",
        "them",
        "their",
        "theirs",
        "he",
        "him",
        "his",
        "she",
        "her",
        "hers",
        "we",
        "us",
        "our",
        "ours",
        "you",
        "your",
        "yours",
        "i",
        "me",
        "my",
        "mine",
        "in",
        "on",
        "at",
        "to",
        "from",
        "with",
        "without",
        "for",
        "of",
        "by",
        "over",
        "under",
        "between",
        "through",
        "into",
        "about",
        "against",
        "during",
        "before",
        "after",
        "above",
        "below",
        "is",
        "are",
        "was",
        "were",
        "be",
        "been",
        "being",
        "am",
        "have",
        "has",
        "had",
        "having",
        "do",
        "does",
        "did",
        "doing",
        "done",
        "can",
        "could",
        "will",
        "would",
        "shall",
        "should",
        "may",
        "might",
        "must",
        "not",
        "no",
        "nor",
        "all",
        "any",
        "some",
        "each",
        "every",
        "both",
        "either",
        "neither",
        "other",
        "another",
        "such",
        "same",
        "own",
        "few",
        "little",
        "several",
        "much",
        "many",
        "more",
        "most",
        "less",
        "least",
        "also",
        "just",
        "only",
        "even",
        "still",
        "yet",
        "already",
        "always",
        "never",
        "often",
        "sometimes",
        "usually",
        "rarely",
        "here",
        "there",
        "when",
        "where",
        "why",
        "how",
        "what",
        "whether",
        "because",
        "therefore",
        "thus",
        "hence",
        "consequently",
        "moreover",
        "furthermore",
        "however",
        "nevertheless",
        "nonetheless",
        "now",
        "then",
        "since",
        "until",
    ]
)


def tokenize(text: str, lower: bool = True) -> list[str]:
    """Split a passage into alphabetic word tokens."""
    words = _TOKEN_RE.findall(text)
    return [w.lower() for w in words] if lower else words


def log_odds_ratio(
    counts_a: np.ndarray,
    counts_b: np.ndarray,
    prior_a: np.ndarray | None = None,
    prior_b: np.ndarray | None = None,
    a0: float = 500.0,
) -> tuple[np.ndarray, np.ndarray]:
    """Monroe et al. log-odds ratio and its z-score.

    Parameters
    ----------
    counts_a, counts_b:
        Raw token counts for the two corpora.
    prior_a, prior_b:
        Informative priors (usually the corpora's own token distributions).
        Defaults to uniform.
    a0:
        Strength of the prior in pseudo-tokens.

    Returns
    -------
    (delta, z):
        ``delta`` is the log-odds of the word appearing in corpus A rather
        than B; ``z`` is that effect size divided by its standard error and
        is the quantity worth ranking on.
    """
    counts_a = np.asarray(counts_a, dtype=float)
    counts_b = np.asarray(counts_b, dtype=float)
    n_a, n_b = counts_a.sum(), counts_b.sum()
    vocab = counts_a.size

    if prior_a is None:
        prior_a = np.full(vocab, 1.0 / vocab)
    if prior_b is None:
        prior_b = np.full(vocab, 1.0 / vocab)

    p_a = (counts_a + a0 * prior_a) / (n_a + a0)
    p_b = (counts_b + a0 * prior_b) / (n_b + a0)
    p_a = np.clip(p_a, 1e-12, 1 - 1e-12)
    p_b = np.clip(p_b, 1e-12, 1 - 1e-12)

    delta = np.log(p_a / p_b) - np.log((1 - p_a) / (1 - p_b))
    var = 1.0 / (n_a * p_a * (1 - p_a)) + 1.0 / (n_b * p_b * (1 - p_b))
    z = delta / np.sqrt(var)
    return delta, z


def _counts_by_label(
    texts: Sequence[str],
    labels: np.ndarray,
    inside_labels: set[str],
    stop: set[str],
) -> tuple[Counter, Counter]:
    """Document-frequency counts split into "inside" and "outside" corpora."""
    inside: Counter = Counter()
    outside: Counter = Counter()
    for text, label in zip(texts, labels, strict=False):
        toks = {t for t in tokenize(text) if t not in stop}
        (inside if label in inside_labels else outside).update(toks)
    return inside, outside


def _rank(
    inside: Counter,
    outside: Counter,
    top_n: int,
    min_count: int,
) -> pd.DataFrame:
    vocab = sorted(w for w in (set(inside) | set(outside)) if inside[w] + outside[w] >= min_count)
    if not vocab:
        return pd.DataFrame(columns=["word", "log_odds", "z"])
    a = np.array([inside[w] for w in vocab], dtype=float)
    b = np.array([outside[w] for w in vocab], dtype=float)
    prior_a = (a + 1.0) / (a.sum() + len(a))
    prior_b = (b + 1.0) / (b.sum() + len(b))
    delta, z = log_odds_ratio(a, b, prior_a, prior_b)
    order = np.argsort(z)[::-1][:top_n]
    return pd.DataFrame(
        {
            "word": [vocab[i] for i in order],
            "log_odds": np.round(delta[order], 2),
            "z": np.round(z[order], 1),
        }
    )


def distinctive_words(
    texts: Sequence[str],
    labels: np.ndarray,
    target: str,
    top_n: int = 12,
    min_count: int = 20,
    exclude: Iterable[str] | None = None,
) -> pd.DataFrame:
    """Most characteristic words of *target* versus every other model.

    Returns a DataFrame with columns ``word``, ``log_odds`` and ``z``.
    """
    stop = set(exclude) if exclude is not None else set(STOPWORDS)
    inside, outside = _counts_by_label(texts, labels, {target}, stop)
    return _rank(inside, outside, top_n, min_count)


def distinctive_profile(
    texts: Sequence[str],
    labels: np.ndarray,
    targets: Sequence[str],
    top_n: int = 12,
    min_count: int = 20,
    exclude: Iterable[str] | None = None,
) -> dict[str, pd.DataFrame]:
    """Run :func:`distinctive_words` for several targets at once."""
    return {
        target: distinctive_words(
            texts, labels, target, top_n=top_n, min_count=min_count, exclude=exclude
        )
        for target in targets
    }


def family_distinctive_words(
    texts: Sequence[str],
    labels: np.ndarray,
    families: dict[str, str],
    family: str,
    top_n: int = 12,
    min_count: int = 20,
    exclude: Iterable[str] | None = None,
) -> pd.DataFrame:
    """Words used more by every model of one lab than by any other lab.

    A family profile is the readable summary of "which lab is this" — and,
    measured on the train set, it is a far stronger signal than any
    individual model's profile.
    """
    stop = set(exclude) if exclude is not None else set(STOPWORDS)
    members = {m for m, f in families.items() if f == family}
    inside, outside = _counts_by_label(texts, labels, members, stop)
    return _rank(inside, outside, top_n, min_count)


def top_content_words(
    texts: Sequence[str],
    labels: np.ndarray,
    target: str,
    top_n: int = 15,
    exclude: Iterable[str] | None = None,
) -> pd.DataFrame:
    """Most repeated content words of a model (an absolute view, not relative)."""
    stop = set(exclude) if exclude is not None else set(STOPWORDS)
    counter: Counter = Counter()
    for text, label in zip(texts, labels, strict=False):
        if label == target:
            counter.update(set(tokenize(text)) - stop)
    frame = pd.DataFrame({"word": list(counter), "count": list(counter.values())}).sort_values(
        "count", ascending=False
    )
    return frame.head(top_n).reset_index(drop=True)
# %% module: groups
"""Recovering the prompt structure from the passages alone.

The train set tells us which passages answered the same prompt
(``prompt_id``).  The hidden test set does not — it is just a bag of
passages.  But the *transductive* structure is still there: passages that
answer the same prompt talk about the same thing, so in content-word TF-IDF
cosine they sit much closer to each other than to passages answering
different prompts.  Measured on train, a passage's nearest neighbour shares
its prompt 99.8% of the time, while the average same-prompt distance is
0.785 against 0.989 for different prompts.

So the grouping can be *inferred*:

1. vectorise passages by content words only (the topic, which is what makes
   same-prompt passages alike);
2. build a k-nearest-neighbour graph over cosine distance;
3. keep edges below a distance cut;
4. take connected components.

Why a k-NN graph rather than plain single-linkage clustering?  Single
linkage chains: one spuriously close pair of passages merges two whole
prompts into one cluster, and the error then propagates.  Requiring k
neighbours to agree suppresses those bridges — but only if the distance cut
is tight enough.

**The cut has to be chosen at the scale the grouping will be used.**  On a
540-passage cross-validation fold a cut of 0.95 looks perfect (ARI 1.00,
because the pool is small enough that nothing chains).  On the full
2,699-passage pool the same cut merges prompts: ARI falls to 0.58 and one
cluster swallows 252 passages.  A cut of 0.88 is the setting that holds up
at every pool size tried (ARI 0.97 at 2,699 passages, 0.96-0.98 from 180 to
4,000 passages).

The local evidence is scale-free, by contrast: at *every* pool size, 99.8%
of nearest neighbours share a prompt.  The structure is there; only the
clustering step was fragile.
"""

from __future__ import annotations

from collections.abc import Sequence

import numpy as np
from scipy import sparse
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import adjusted_rand_score


def content_distance_matrix(texts: Sequence[str], stop_words: str = "english") -> np.ndarray:
    """Pairwise cosine distance in content-word TF-IDF space."""
    vec = TfidfVectorizer(stop_words=stop_words, sublinear_tf=True, dtype=np.float32)
    X = vec.fit_transform(texts)
    similarity = (X @ X.T).toarray()
    np.fill_diagonal(similarity, 0.0)
    return np.clip(1.0 - similarity, 0.0, 1.0)


def infer_groups(
    texts: Sequence[str],
    k: int = 3,
    cut: float = 0.88,
    distance: np.ndarray | None = None,
) -> np.ndarray:
    """Cluster passages into recovered prompt groups.

    Parameters
    ----------
    texts:
        The passages of the pool (train fold, or the whole test set).
    k:
        Number of nearest neighbours that must agree before an edge is kept.
    cut:
        Maximum cosine distance for a kept edge.  0.88 is the value that
        stays accurate as the pool grows — see the module docstring.
    distance:
        Pre-computed distance matrix (to avoid recomputing it).

    Returns
    -------
    Array of group ids, one per passage.
    """
    n = len(texts)
    if n < 2:
        return np.zeros(n, dtype=int)

    D = content_distance_matrix(texts) if distance is None else distance
    order = np.argsort(D, axis=1)
    rows = np.repeat(np.arange(n), k)
    cols = order[:, :k].ravel()
    keep = D[rows, cols] < cut

    graph = sparse.csr_matrix((np.ones(int(keep.sum())), (rows[keep], cols[keep])), shape=(n, n))
    _, labels = connected_components(graph, directed=False)
    return labels


def group_sizes(groups: np.ndarray) -> np.ndarray:
    """Number of passages in each recovered group, sorted descending."""
    _, counts = np.unique(groups, return_counts=True)
    return np.sort(counts)[::-1]


def neighbour_purity(distance: np.ndarray, true_groups: np.ndarray, k: int = 5) -> float:
    """Share of nearest neighbours that share the passage's true group."""
    n = len(true_groups)
    if n < 2:
        return float("nan")
    order = np.argsort(distance, axis=1)
    same = 0
    for i in range(n):
        for j in order[i, :k]:
            if true_groups[i] == true_groups[j]:
                same += 1
    return same / (n * k)


def group_report(groups: np.ndarray, true_groups: np.ndarray | None = None) -> dict:
    """Summarise a recovered grouping.

    With ``true_groups`` present, reports the adjusted Rand index (does the
    partition match?) plus the number of recovered groups and their size
    histogram.
    """
    sizes = group_sizes(groups)
    report = {
        "n_groups": int(len(sizes)),
        "largest_group": int(sizes[0]) if len(sizes) else 0,
        "smallest_group": int(sizes[-1]) if len(sizes) else 0,
        "mean_size": float(sizes.mean()) if len(sizes) else 0.0,
        "n_singletons": int((sizes == 1).sum()),
    }
    if true_groups is not None:
        report["ari"] = float(adjusted_rand_score(true_groups, groups))
        report["n_true_groups"] = int(len(np.unique(true_groups)))
        # purity: share of passages sitting in a group dominated by their own label
        correct = 0
        for g in np.unique(groups):
            members = true_groups[groups == g]
            values, counts = np.unique(members, return_counts=True)
            correct += counts.max()
        report["purity"] = correct / len(true_groups)
    return report
# %% module: decision
"""Decision rules — turning scores into labels, and what the metric wants.

The competition metric is **balanced accuracy**, the mean of per-class
recalls::

    balanced_accuracy = (1/K) * sum_k  #{i : y_i = k and yhat_i = k} / N_k

where ``N_k`` is the number of test passages whose true label is ``k``.  Two
consequences drive everything in this module.

**1. What is the optimal rule for one passage?**
Write the metric as a sum over passages::

    (1/K) * sum_i sum_k P(y_i = k | x_i) * 1[yhat_i = k] / N_k

Every passage can be labelled independently, and the expected contribution of
labelling passage *i* with class *j* is ``P(y_i = j | x_i) / N_j``.  So the
metric-optimal choice is::

    yhat_i = argmax_j P(y_i = j | x_i) / N_j

If the classifier was trained on a *uniform* class prior (this train set is
uniform: 150 passages per model) and the test prior is ``pi_test``, then
``P(y = j | x) ∝ p_source(j | x) * pi_test_j / pi_source_j`` and ``N_j`` is
proportional to ``pi_test_j``.  The prior terms cancel::

    yhat_i = argmax_j p_source(j | x_i) / pi_source_j

With a uniform source prior that is simply **plain argmax** of the model
scores, whatever the test class mix happens to be.  Two corollaries, both
verified empirically on the train set:

* *Do not* try to reproduce the true test class mix (Sinkhorn towards the
  observed mix): it costs a lot of accuracy under any skew.
* *Do not* divide by an estimated test prior either: that is the correction
  applied in the wrong direction, and it is worse still.

The train prior being uniform is doing the work here.  That is worth stating
plainly, because it is the reason the pipeline deliberately does **not**
rebalance its output.

**2. What about the structure inside a prompt?**
Independent argmax ignores a piece of hard information: in the train set
each prompt is answered by each of the 18 models exactly once.  Inside a
recovered group the labels should therefore be a *permutation*, not 18
independent draws.  Enforcing that turns 18 independent decisions into one
joint one and is worth roughly +0.10 balanced accuracy.  The constraint is
about the *design of the data*, not about the class mix — and it is applied
as "at most one passage per model per group", which stays valid even if the
hidden test is not perfectly balanced (measured: it still helps, by less).
"""

from __future__ import annotations

from collections.abc import Sequence

import numpy as np
from scipy.optimize import linear_sum_assignment


def argmax_decision(scores: np.ndarray, classes: Sequence[str]) -> np.ndarray:
    """Plain argmax — the metric-optimal rule for a uniform source prior."""
    return np.asarray(classes, dtype=object)[np.asarray(scores).argmax(axis=1)]


def sinkhorn(
    probabilities: np.ndarray, target: np.ndarray | None = None, iters: int = 20
) -> np.ndarray:
    """Iterative proportional fitting of the column marginals to *target*.

    With ``target=None`` this pushes the predicted class mass towards a
    uniform mix.  It is a *heuristic* about the output distribution, not a
    consequence of balanced accuracy — see the module docstring.  It was
    measured to give a small, mix-robust improvement over plain argmax, but
    it is not what the metric asks for, so it is opt-in.
    """
    P = np.array(probabilities, dtype=float, copy=True)
    n, k = P.shape
    if target is None:
        target = np.full(k, n / k)
    target = np.asarray(target, dtype=float)
    for _ in range(iters):
        P *= target / np.maximum(P.sum(axis=0), 1e-12)
        P /= np.maximum(P.sum(axis=1, keepdims=True), 1e-12)
    return P


def em_prior(
    probabilities: np.ndarray,
    source_prior: np.ndarray | None = None,
    iters: int = 50,
    tol: float = 1e-9,
) -> np.ndarray:
    """Saerens-Latinne-Decaestecker estimate of the *test* class prior.

    E-step: re-weight the source posterior by the prior shift.
    M-step: the prior is the mean re-weighted posterior.

    Included because it is the standard tool for this question — and
    included with a warning: for **balanced accuracy** the test prior is
    *not* what the decision rule should divide by (the module docstring
    derives why).  Applying it costs most of the accuracy.
    """
    P = np.asarray(probabilities, dtype=float)
    k = P.shape[1]
    src = np.full(k, 1.0 / k) if source_prior is None else np.asarray(source_prior, float)
    pi = src.copy()
    for _ in range(iters):
        W = P * (pi / src)
        W /= np.maximum(W.sum(axis=1, keepdims=True), 1e-12)
        new = W.mean(axis=0)
        if np.abs(new - pi).max() < tol:
            pi = new
            break
        pi = new
    return pi


def assign_within_groups(
    scores: np.ndarray,
    groups: np.ndarray,
    classes: Sequence[str],
    at_most_one: bool = True,
) -> np.ndarray:
    """Assign labels inside each group so that no model repeats (if possible).

    Parameters
    ----------
    scores:
        ``(n_passages, n_classes)`` score matrix (higher is better).
    groups:
        Group id per passage.
    classes:
        Class names, in the order of ``scores``' columns.
    at_most_one:
        When True (default) a model may be used at most once per group.  This
        is the relaxation that stays valid even if the hidden test does not
        contain exactly one passage per model per prompt.  When False every
        passage is labelled independently by argmax.

    Returns
    -------
    Array of labels, one per passage.
    """
    scores = np.asarray(scores, dtype=float)
    classes = np.asarray(classes, dtype=object)
    groups = np.asarray(groups)
    n_classes = scores.shape[1]
    labels = np.empty(len(scores), dtype=object)

    for group in np.unique(groups):
        idx = np.flatnonzero(groups == group)
        block = scores[idx]
        if not at_most_one or len(idx) == 1:
            labels[idx] = classes[block.argmax(axis=1)]
            continue
        if len(idx) <= n_classes:
            rows, cols = linear_sum_assignment(-block)
            assigned = np.empty(len(idx), dtype=object)
            assigned[rows] = classes[cols]
            labels[idx] = assigned
        else:
            # More passages than models: by pigeonhole some model must repeat.
            # Pad with dummy columns so every passage still receives a label
            # and real models stay unique; passages forced onto a dummy column
            # keep their own best guess.
            padding = np.full((len(idx), len(idx) - n_classes), -1e9)
            padded = np.concatenate([block, padding], axis=1)
            rows, cols = linear_sum_assignment(-padded)
            fallback = classes[block.argmax(axis=1)]
            chosen = np.where(cols < n_classes, classes[np.clip(cols, 0, n_classes - 1)], fallback)
            assigned = np.empty(len(idx), dtype=object)
            assigned[rows] = chosen[rows]
            labels[idx] = assigned
    return labels


def balanced_accuracy(y_true: Sequence[str], y_pred: Sequence[str]) -> float:
    """Mean per-class recall — the competition metric."""
    from sklearn.metrics import balanced_accuracy_score

    return float(balanced_accuracy_score(list(y_true), list(y_pred)))
# %% module: pipeline
"""The stack: views -> prompt centring -> classifier -> group assignment.

Each layer exists because of a specific, measured failure of the layer
below it:

``views``
    Raw word n-grams spend most of their capacity on *topic*.  Adding
    topic-free views (function words, word shape) lifts group-CV balanced
    accuracy from 0.31 to 0.36.

``centring``
    Even the best view is still mostly measuring the prompt.  Subtracting
    the group mean removes the shared component, because what all 18 answers
    to one prompt have in common is the prompt, not the author.

``classifier``
    A linear model over the (centred) TF-IDF space.  Deliberately simple:
    it is the part of the system we understand best.

``assignment``
    Independent per-passage argmax throws away the knowledge that a prompt
    is answered by each model once.  Solving a small assignment problem
    inside each recovered group enforces it.

The layers are additive in the measurements that motivated them, and each
one can be switched off to reproduce the ablation.
"""

from __future__ import annotations

from collections.abc import Sequence
from dataclasses import dataclass

import numpy as np
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.svm import LinearSVC

from .decision import assign_within_groups, sinkhorn
from .facets import facet_frame
from .groups import infer_groups
from .views import DEFAULT_VIEWS, fit_views, transform_views


@dataclass
class StackConfig:
    """Every knob of the stack, so that an ablation is one argument away."""

    views: tuple[str, ...] = DEFAULT_VIEWS
    C: float = 4.0
    center: bool = True
    assign: bool = True
    balance: bool = False
    group_k: int = 3
    group_cut: float = 0.88
    center_min_size: int = 2
    center_max_size: int | None = 18
    max_iter: int = 3000
    classifier: str = "logreg"
    random_state: int = 42
    #: "pool" groups the *whole* set of passages at once (what happens at
    #: test time); "fold" groups each cross-validation fold separately.
    grouping_scope: str = "pool"
    #: Numeric facet families (or individual columns) appended as one extra
    #: block.  See :mod:`ntl.facets`.  Empty means "views only".
    facets: tuple[str, ...] = ()
    #: Relative weight of the facet block against one view block.  Each block
    #: is row-normalised, so 1.0 makes the facets worth one view.
    facet_weight: float = 1.0
    #: Subtract the group mean from the facet block as well (the same
    #: "the prompt is not the author" move, applied to the numbers).
    facet_center: bool = True


def center_within(
    X: sparse.spmatrix,
    groups: np.ndarray,
    min_size: int = 2,
    max_size: int | None = 18,
) -> sparse.csr_matrix:
    """Subtract the mean of each group from every member of that group.

    This is the "the prompt is not the author" move made concrete: the mean
    of all the answers to one prompt is, to a first approximation, the
    prompt itself.  Removing it leaves the *residual* — what each model
    added on top of the shared material.

    Groups outside ``[min_size, max_size]`` are left untouched: a group of
    one carries no mean worth removing, and a group that merged two prompts
    would have its (meaningful) mean removed as well.
    """
    X = sparse.csr_matrix(X)
    groups = np.asarray(groups)
    _, inverse, counts = np.unique(groups, return_inverse=True, return_counts=True)
    keep = np.flatnonzero(counts >= min_size)
    if max_size is not None:
        keep = keep[counts[keep] <= max_size]
    if not len(keep):
        return X.tocsr()

    column = np.full(len(counts), -1)
    column[keep] = np.arange(len(keep))
    rows = np.flatnonzero(column[inverse] >= 0)
    membership = sparse.csr_matrix(
        (np.ones(len(rows)), (rows, column[inverse][rows])),
        shape=(X.shape[0], len(keep)),
    )
    means = sparse.diags(1.0 / counts[keep]) @ (membership.T @ X)
    return (X - membership @ means).tocsr()


def center_dense_within(
    X: np.ndarray,
    groups: np.ndarray,
    min_size: int = 2,
    max_size: int | None = 18,
) -> np.ndarray:
    """Dense counterpart of :func:`center_within`, for numeric facet blocks.

    Same rule, same exceptions: groups smaller than *min_size* or larger than
    *max_size* keep their raw values.  A group of one has no mean worth
    removing; an oversized group is probably two prompts merged, and its mean
    is therefore meaningful rather than nuisance.
    """
    out = np.array(X, dtype=float, copy=True)
    groups = np.asarray(groups)
    _, inverse, counts = np.unique(groups, return_inverse=True, return_counts=True)
    for index in np.flatnonzero(counts >= min_size):
        if max_size is not None and counts[index] > max_size:
            continue
        rows = np.flatnonzero(inverse == index)
        out[rows] -= out[rows].mean(axis=0, keepdims=True)
    return out


class StackModel:
    """The full pipeline, fit on labelled passages and applied to a pool."""

    def __init__(self, config: StackConfig | None = None):
        self.config = config or StackConfig()
        self.classes_: np.ndarray | None = None
        self.vecs_: dict | None = None
        self.clf_: object | None = None
        self.scaler_: StandardScaler | None = None

    # -- fitting -----------------------------------------------------------
    def _make_classifier(self):
        cfg = self.config
        if cfg.classifier == "linearsvc":
            return LinearSVC(
                C=cfg.C,
                class_weight="balanced",
                max_iter=5000,
                random_state=cfg.random_state,
            )
        return LogisticRegression(
            C=cfg.C,
            max_iter=cfg.max_iter,
            class_weight="balanced",
            random_state=cfg.random_state,
        )

    # -- feature assembly --------------------------------------------------
    def _facet_block(
        self, texts: Sequence[str], groups: np.ndarray, fit: bool
    ) -> sparse.csr_matrix:
        """Standardise, row-normalise and (optionally) centre the numeric block."""
        cfg = self.config
        frame = facet_frame(texts, groups, cfg.facets).to_numpy(dtype=float)
        frame = np.nan_to_num(frame, nan=0.0, posinf=0.0, neginf=0.0)
        if fit:
            self.scaler_ = StandardScaler().fit(frame)
        assert self.scaler_ is not None, "fit the facet scaler first"
        block = normalize(self.scaler_.transform(frame))
        if cfg.facet_center and not cfg.center:
            block = center_dense_within(block, groups, cfg.center_min_size, cfg.center_max_size)
        return sparse.csr_matrix(block * cfg.facet_weight)

    def _feature_matrix(
        self, texts: Sequence[str], groups: np.ndarray | None, fit: bool
    ) -> sparse.csr_matrix:
        cfg = self.config
        X = transform_views(self.vecs_, texts, cfg.views)
        if cfg.facets:
            assert groups is not None, "facets need a grouping"
            X = sparse.hstack([X, self._facet_block(texts, groups, fit)], format="csr")
        if cfg.center:
            assert groups is not None, "centring needs a grouping"
            X = center_within(X, groups, cfg.center_min_size, cfg.center_max_size)
        return X

    def fit(
        self,
        texts: Sequence[str],
        y: Sequence[str],
        groups: Sequence | None = None,
    ) -> StackModel:
        """Fit the stack.

        *groups* gives the known prompt grouping of the training passages
        (the train set's ``prompt_id`` column).  When omitted, the grouping
        is inferred from the texts exactly as it is at prediction time.
        """
        cfg = self.config
        self.vecs_ = fit_views(texts, cfg.views)
        if groups is None and (cfg.center or cfg.facets):
            groups = infer_groups(texts, k=cfg.group_k, cut=cfg.group_cut)
        X = self._feature_matrix(texts, None if groups is None else np.asarray(groups), fit=True)
        self.clf_ = self._make_classifier()
        self.clf_.fit(X, np.asarray(y, dtype=object))
        self.classes_ = np.asarray(self.clf_.classes_, dtype=object)
        return self

    # -- prediction --------------------------------------------------------
    def decision_function(
        self, texts: Sequence[str], groups: np.ndarray | None = None
    ) -> tuple[np.ndarray, np.ndarray]:
        """Scores for every class, plus the group id of every passage."""
        cfg = self.config
        assert self.vecs_ is not None and self.clf_ is not None, "fit first"
        if groups is None:
            groups = infer_groups(texts, k=cfg.group_k, cut=cfg.group_cut)
        groups = np.asarray(groups)
        X = self._feature_matrix(texts, groups, fit=False)
        # The classifier's own class order is the column order of ``classes_``
        # (both come from ``clf_.classes_``), so the scores need no remapping.
        scores = np.asarray(self.clf_.decision_function(X), dtype=float)
        if scores.ndim == 1:  # binary fallback
            scores = np.column_stack([-scores, scores])
        return scores, groups

    def predict(self, texts: Sequence[str], groups: np.ndarray | None = None) -> np.ndarray:
        """Label every passage in the pool."""
        cfg = self.config
        scores, groups = self.decision_function(texts, groups)
        if cfg.balance:
            probs = _softmax(scores)
            scores = np.log(np.maximum(sinkhorn(probs), 1e-12))
        if cfg.assign:
            return assign_within_groups(scores, groups, self.classes_)
        return self.classes_[scores.argmax(axis=1)]


def _softmax(scores: np.ndarray) -> np.ndarray:
    shifted = scores - scores.max(axis=1, keepdims=True)
    exp = np.exp(np.clip(shifted, -60, 0))
    return exp / exp.sum(axis=1, keepdims=True)


def group_cv_score(
    texts: Sequence[str],
    y: Sequence[str],
    groups: Sequence,
    config: StackConfig | None = None,
    n_splits: int = 5,
    return_oof: bool = False,
) -> dict:
    """Out-of-fold balanced accuracy under ``GroupKFold`` on the prompt ids.

    Random cross-validation is *not* a proxy for the leaderboard here — it is
    a different and much harder task.  Siblings of a validation passage (the
    other answers to the same prompt) sit in the training fold carrying
    *different* labels, so the model is asked to map one piece of content to
    18 different authors; the cheapest way to fit that is to ignore the
    content, and with it the style.  Grouping by prompt removes the
    contradiction and lets the model use style instead.

    The *validation* grouping is recovered from text alone, never from
    ``groups``.  With ``config.grouping_scope == "pool"`` the recovery runs
    over the whole set of passages, exactly as it will over the whole test
    pool — which matters, because grouping quality depends on pool size
    (see :mod:`ntl.groups`).
    """
    cfg = config or StackConfig()
    y = np.asarray(y, dtype=object)
    groups = np.asarray(groups)
    oof = np.empty(len(y), dtype=object)
    folds: list[float] = []

    pool_groups = None
    if cfg.grouping_scope == "pool" and (cfg.center or cfg.assign):
        pool_groups = infer_groups(texts, k=cfg.group_k, cut=cfg.group_cut)

    splitter = GroupKFold(n_splits=n_splits)
    for train_idx, valid_idx in splitter.split(texts, y, groups):
        model = StackModel(cfg)
        model.fit(
            [texts[i] for i in train_idx],
            y[train_idx],
            groups=groups[train_idx],
        )
        if pool_groups is not None:
            valid_groups = pool_groups[valid_idx]
        elif cfg.center or cfg.assign:
            valid_groups = infer_groups(
                [texts[i] for i in valid_idx], k=cfg.group_k, cut=cfg.group_cut
            )
        else:
            valid_groups = np.zeros(len(valid_idx), dtype=int)
        pred = model.predict([texts[i] for i in valid_idx], groups=valid_groups)
        oof[valid_idx] = pred
        folds.append(balanced_accuracy_score(y[valid_idx], pred))

    result = {
        "balanced_accuracy": float(balanced_accuracy_score(y, oof)),
        "fold_scores": folds,
        "fold_mean": float(np.mean(folds)),
        "fold_std": float(np.std(folds)),
    }
    if return_oof:
        result["oof"] = oof
    return result
# %% module: measure
"""Which facets actually carry author signal?

A facet can look informative for the wrong reason.  Response length, for
example, has a large mutual information with the model label on the train set
— because some prompts ask for an essay and others for a sentence, and the
prompt mix is not uniform across models by accident.  That is a *prompt
proxy*, not an author habit, and it will not transfer to a hidden prompt.

The honest question is: **after the prompt is held constant, does the facet
still vary with the model?**  This module answers it with two statistics and
one classifier.

``eta2_within``
    Centre each prompt group on its own mean, then ask how much of the
    remaining variance the model label explains (a one-way ANOVA on the
    residuals).  Near zero means the facet is a prompt proxy; clearly above
    zero means it separates siblings writing about the same thing.

``mi``
    Raw mutual information with the label.  Reported only as a contrast: a
    facet with high ``mi`` and low ``eta2_within`` is the trap described
    above.  It is never used for selection.

``facet_cv_score``
    Prompt-grouped cross-validation with the facet block alone.  This is the
    end-to-end check: does the signal survive a fitted classifier?

``infer_domains``
    Clusters recovered-group centroids into topics.  Included to *demonstrate*
    the prompt-side constraint rather than to build a feature: the inferred
    domain is constant inside every recovered group, so it cannot separate
    siblings (see :func:`group_constant_share`).
"""

from __future__ import annotations

from collections.abc import Sequence

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler, normalize

from . import facets
from .pipeline import center_dense_within

#: ``eta2_within`` below this is treated as "no author signal".  It is a
#: reporting threshold, not a fitted parameter: the notebook prints the whole
#: ranking so the reader can disagree with the cut.
DEFAULT_SIGNAL_FLOOR = 0.005


def within_group_eta2(
    values: Sequence[float],
    y: Sequence[str],
    groups: Sequence,
    min_group: int = 2,
) -> float:
    """Share of within-group variance explained by the model label.

    1. subtract each group's mean (removing the prompt);
    2. run a one-way ANOVA of the residuals by model;
    3. return between-model SS / total SS.

    Groups smaller than *min_group* are left in place: they contribute no
    within-group variation, so removing them would silently flatter the
    result.
    """
    v = np.asarray(values, dtype=float)
    y = np.asarray(y, dtype=object)
    groups = np.asarray(groups)
    if len(v) < 3:
        return 0.0

    frame = pd.DataFrame({"v": v, "g": groups})
    centred = v - frame.groupby("g")["v"].transform("mean").to_numpy(dtype=float)
    total = float(np.sum((centred - centred.mean()) ** 2))
    if total < 1e-12:
        return 0.0

    between = 0.0
    for label in np.unique(y):
        mask = y == label
        if mask.sum() == 0:
            continue
        between += float(mask.sum()) * float(centred[mask].mean() ** 2)
    return float(np.clip(between / total, 0.0, 1.0))


def mutual_information(values: Sequence[float], y: Sequence[str]) -> float:
    """Raw mutual information between a facet and the label.

    Discretisation is automatic.  High values are *not* evidence of an author
    habit — see the module docstring.
    """
    v = np.asarray(values, dtype=float).reshape(-1, 1)
    v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)
    return float(mutual_info_classif(v, np.asarray(y, dtype=object), random_state=0)[0])


def group_constant_share(values: Sequence[float], groups: Sequence) -> float:
    """Share of groups in which the facet takes a single value.

    A facet at 1.0 is constant inside every prompt group and therefore
    *cannot* identify the author no matter how large its raw mutual
    information is.  Domain labels and prompt-level axes land here.
    """
    v = np.asarray(values)
    groups = np.asarray(groups)
    unique = np.unique(groups)
    if not len(unique):
        return 0.0
    constant = 0
    for group in unique:
        if len(np.unique(v[groups == group])) <= 1:
            constant += 1
    return constant / len(unique)


def _mean_within_sd(values: np.ndarray, groups: np.ndarray) -> float:
    sds = []
    for group in np.unique(groups):
        block = values[groups == group]
        if len(block) >= 2:
            sds.append(float(np.std(block)))
    return float(np.mean(sds)) if sds else 0.0


def facet_signal(
    texts: Sequence[str],
    y: Sequence[str],
    groups: Sequence,
    signal_floor: float = DEFAULT_SIGNAL_FLOOR,
) -> pd.DataFrame:
    """One row per facet: ``mi``, ``eta2_within``, and a verdict.

    The frame is sorted by ``eta2_within`` descending, so the first rows are
    the facets worth putting in a classifier.
    """
    y = np.asarray(y, dtype=object)
    groups = np.asarray(groups)
    base = facets.response_facets(texts)
    frame = pd.concat([base, facets.group_facets(texts, groups, base)], axis=1)

    rows = []
    for column in frame.columns:
        values = np.nan_to_num(frame[column].to_numpy(dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
        eta2 = within_group_eta2(values, y, groups)
        rows.append(
            {
                "facet": column,
                "family": facets.family_of(column),
                "mi": mutual_information(values, y),
                "eta2_within": eta2,
                "constant_share": group_constant_share(values, groups),
                "within_sd": _mean_within_sd(values, groups),
                "verdict": "author signal" if eta2 >= signal_floor else "prompt proxy",
            }
        )
    out = pd.DataFrame(rows)
    return out.sort_values("eta2_within", ascending=False).reset_index(drop=True)


def _facet_matrix(texts: Sequence[str], groups: np.ndarray, families: Sequence[str]) -> np.ndarray:
    frame = facets.facet_frame(texts, groups, families).to_numpy(dtype=float)
    return np.nan_to_num(frame, nan=0.0, posinf=0.0, neginf=0.0)


def facet_cv_score(
    texts: Sequence[str],
    y: Sequence[str],
    groups: Sequence,
    families: Sequence[str],
    n_splits: int = 5,
    C: float = 4.0,
    center: bool = True,
) -> dict:
    """Prompt-grouped CV using the facet block alone.

    The numeric block is standardised and (by default) centred within the
    true prompt groups.  Because ``GroupKFold`` never splits a prompt, using
    the true grouping here is the transductive condition, not a leak — at test
    time the grouping is *recovered*, which is slightly noisier (ARI 0.97),
    so treat the result as a mild upper bound on the facet's contribution.
    """
    y = np.asarray(y, dtype=object)
    groups = np.asarray(groups)
    X = _facet_matrix(texts, groups, families)

    splitter = GroupKFold(n_splits=n_splits)
    oof = np.empty(len(y), dtype=object)
    folds: list[float] = []
    for train_idx, valid_idx in splitter.split(X, y, groups):
        scaler = StandardScaler().fit(X[train_idx])
        X_train = scaler.transform(X[train_idx])
        X_valid = scaler.transform(X[valid_idx])
        if center:
            X_train = center_dense_within(X_train, groups[train_idx])
            X_valid = center_dense_within(X_valid, groups[valid_idx])
        clf = LogisticRegression(
            C=C,
            max_iter=3000,
            class_weight="balanced",
            random_state=42,
        )
        clf.fit(X_train, y[train_idx])
        pred = clf.predict(X_valid)
        oof[valid_idx] = pred
        folds.append(float(balanced_accuracy_score(y[valid_idx], pred)))

    return {
        "balanced_accuracy": float(balanced_accuracy_score(y, oof)),
        "fold_scores": folds,
        "fold_mean": float(np.mean(folds)),
        "fold_std": float(np.std(folds)),
    }


def family_ranking(
    texts: Sequence[str],
    y: Sequence[str],
    groups: Sequence,
    families: Sequence[str] | None = None,
    n_splits: int = 5,
    C: float = 4.0,
) -> pd.DataFrame:
    """Grouped CV for each facet family on its own, best first."""
    if families is None:
        families = [*facets.RESPONSE_FAMILIES, "group"]
    rows = []
    for family in families:
        score = facet_cv_score(texts, y, groups, [family], n_splits=n_splits, C=C)
        rows.append(
            {
                "family": family,
                "n_facets": len(facets.facet_columns([family])),
                "balanced_accuracy": score["balanced_accuracy"],
                "fold_mean": score["fold_mean"],
                "fold_std": score["fold_std"],
            }
        )
    return (
        pd.DataFrame(rows).sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
    )


def infer_domains(
    texts: Sequence[str],
    groups: Sequence,
    n_topics: int = 6,
    random_state: int = 42,
) -> dict:
    """Cluster recovered-group centroids into topics.

    Returns ``{"passage_domain": Series, "group_domain": Series,
    "n_topics": int}``.  The point is not the topic model — it is that
    ``group_constant_share(passage_domain, groups) == 1.0``: whatever the
    domain is, every passage in a group gets the same value, so the domain
    cannot separate siblings.  It can only stratify *training*.
    """
    groups = np.asarray(groups)
    n = len(texts)
    unique = np.unique(groups)
    membership = pd.Series(np.arange(len(unique)), index=unique)
    inverse = membership.loc[groups].to_numpy()

    vectorizer = TfidfVectorizer(
        stop_words="english", sublinear_tf=True, min_df=3, dtype=np.float32
    )
    X = vectorizer.fit_transform(texts)
    indicator = sparse.csr_matrix((np.ones(n), (np.arange(n), inverse)), shape=(n, len(unique)))
    centroids = normalize(indicator.T @ X)

    k = int(min(n_topics, len(unique)))
    model = KMeans(n_clusters=k, n_init=10, random_state=random_state)
    group_domain = model.fit_predict(centroids.toarray())
    passage_domain = group_domain[inverse]
    return {
        "passage_domain": pd.Series(passage_domain, name="domain"),
        "group_domain": pd.Series(group_domain, index=unique, name="domain"),
        "n_topics": k,
    }
# %% module: tracker
"""Experiment tracker — the instrument that compares local CV with the LB.

The competition gives five submissions a day and no labels.  The only way to
learn which facet combination works is to record, for each variant, both the
local prompt-grouped CV score and the public leaderboard score, and to watch
the *gap*.  This module is deliberately tiny: a CSV with one row per run, and
a report that keeps the latest row per variant.

A variant's configuration is hashed, so a notebook that was edited (views or
facets changed) cannot silently inherit an old score.

Typical use in a Kaggle notebook::

    from ntl import tracker

    row = tracker.make_row(
        variant="v5",
        views=("char", "word", "funcon"),
        facets=("length", "lexical", "punctuation", "structure", "emoji",
                "person", "hedging", "citation", "opener", "group"),
        local_cv=0.4897,
        local_folds=3,
        n_train=2699,
        n_pool=0,
    )
    print("TRACKER", tracker.to_json(row))
    tracker.append_csv("/kaggle/working/tracker.csv", row)

Then locally::

    uv run python scripts/track.py merge ~/Downloads/tracker.csv
    uv run python scripts/track.py report
"""

from __future__ import annotations

import hashlib
import json
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd

TRACKER_COLUMNS = [
    "variant",
    "config_hash",
    "views",
    "facets",
    "facet_weight",
    "local_cv",
    "local_folds",
    "public_lb",
    "n_train",
    "n_pool",
    "run_at",
    "notes",
]

DEFAULT_PATH = Path("metrics/tracker.csv")


def config_hash(
    views: tuple[str, ...] | list[str],
    facets: tuple[str, ...] | list[str] = (),
    facet_weight: float = 1.0,
) -> str:
    """Short, stable hash of the parts of a variant that change its output."""
    payload = json.dumps(
        {
            "views": list(views),
            "facets": list(facets),
            "facet_weight": float(facet_weight),
        },
        sort_keys=True,
    )
    return hashlib.sha1(payload.encode()).hexdigest()[:10]


def make_row(
    variant: str,
    views: tuple[str, ...] | list[str],
    facets: tuple[str, ...] | list[str] = (),
    facet_weight: float = 1.0,
    local_cv: float | None = None,
    local_folds: int | None = None,
    public_lb: float | None = None,
    n_train: int = 0,
    n_pool: int = 0,
    notes: str = "",
    run_at: str | None = None,
) -> dict:
    """Build one tracker row (all values JSON/CSV friendly)."""
    return {
        "variant": str(variant),
        "config_hash": config_hash(views, facets, facet_weight),
        "views": "+".join(views),
        "facets": "+".join(facets) if facets else "-",
        "facet_weight": float(facet_weight),
        "local_cv": None if local_cv is None else round(float(local_cv), 6),
        "local_folds": local_folds,
        "public_lb": None if public_lb is None else round(float(public_lb), 6),
        "n_train": int(n_train),
        "n_pool": int(n_pool),
        "run_at": run_at or datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "notes": notes,
    }


def to_json(row: dict) -> str:
    """One-line JSON, for copying out of a Kaggle notebook log."""
    return json.dumps(row, sort_keys=True)


def append_csv(path: str | Path, row: dict) -> Path:
    """Append *row* to the tracker CSV, creating it with a header if needed."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame = pd.DataFrame([row]).reindex(columns=TRACKER_COLUMNS)
    header = not path.exists() or path.stat().st_size == 0
    frame.to_csv(path, mode="a", header=header, index=False)
    return path


def load(path: str | Path = DEFAULT_PATH) -> pd.DataFrame:
    """Load the tracker, returning an empty typed frame if it does not exist."""
    path = Path(path)
    if not path.exists():
        return pd.DataFrame(columns=TRACKER_COLUMNS)
    frame = pd.read_csv(path)
    for column in TRACKER_COLUMNS:
        if column not in frame.columns:
            frame[column] = None
    return frame[TRACKER_COLUMNS]


def report(path: str | Path = DEFAULT_PATH) -> pd.DataFrame:
    """Latest run per variant, with the local→public gap.

    ``gap`` is ``public_lb - local_cv``.  A consistently positive gap means
    the pool (or the hidden test being easier than train prompts) is helping;
    a negative gap means the local CV is optimistic.
    """
    frame = load(path)
    if frame.empty:
        return frame.assign(gap=pd.Series(dtype=float))
    frame = frame.sort_values("run_at", kind="stable")
    latest = frame.groupby("variant", as_index=False).tail(1).copy()
    latest["gap"] = latest["public_lb"] - latest["local_cv"]
    return latest.sort_values(
        ["public_lb", "local_cv"], ascending=[False, False], na_position="last"
    ).reset_index(drop=True)


def merge(path: str | Path, other: str | Path) -> int:
    """Append rows from another tracker CSV, skipping exact duplicates.

    Returns the number of rows added.  Used to pull a Kaggle-downloaded
    ``tracker.csv`` into the local record.
    """
    target = Path(path)
    incoming = load(other)
    if incoming.empty:
        return 0
    existing = load(target)
    keys = ["variant", "config_hash", "run_at", "local_cv", "public_lb"]
    known = set(map(tuple, existing[keys].astype(str).itertuples(index=False, name=None)))
    candidates = incoming[
        ~pd.Series(
            list(map(tuple, incoming[keys].astype(str).itertuples(index=False, name=None)))
        ).isin(known)
    ]
    if candidates.empty:
        return 0
    target.parent.mkdir(parents=True, exist_ok=True)
    header = not target.exists() or target.stat().st_size == 0
    candidates.to_csv(target, mode="a", header=header, index=False)
    return int(len(candidates))
# %% module: explore
"""Exploratory analysis helpers.

Functions here return data (DataFrames / Series / dicts) rather than plotting,
so they compose cleanly in notebooks and stay headless-friendly. Charts live
in :mod:`ntl.plots`.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from .config import LABELS, PROMPT_COL, PROVIDERS, TARGET_COL, TEXT_COL
from .features import FEATURE_COLS, add_text_features
from .groups import group_report, infer_groups
from .stylometry import (
    STOPWORDS,
    distinctive_profile,
    family_distinctive_words,
)
from .views import VIEWS, view_text


def label_distribution(train: pd.DataFrame) -> pd.DataFrame:
    """Per-model passage counts, share, and provider."""
    counts = train[TARGET_COL].value_counts().reindex(LABELS, fill_value=0)
    out = pd.DataFrame(
        {
            "model": counts.index,
            "n_passages": counts.values,
            "share": counts.values / counts.sum(),
            "provider": [PROVIDERS[m] for m in counts.index],
        }
    )
    return out.reset_index(drop=True)


def prompt_stats(train: pd.DataFrame) -> pd.DataFrame:
    """Passages and distinct models per prompt_id."""
    g = train.groupby(PROMPT_COL)
    out = pd.DataFrame(
        {
            "n_passages": g.size(),
            "n_models": g[TARGET_COL].nunique(),
        }
    )
    return out.sort_values("n_passages", ascending=False)


def per_model_features(train: pd.DataFrame) -> pd.DataFrame:
    """Stylistic features aggregated per model (mean)."""
    feats = add_text_features(train)
    return feats.groupby(TARGET_COL)[FEATURE_COLS].mean().reindex(LABELS)


def feature_zscores_per_model(train: pd.DataFrame) -> pd.DataFrame:
    """Per-model mean features, z-scored across models — heatmap-friendly."""
    means = per_model_features(train)
    return (means - means.mean()) / means.std(ddof=0).replace(0, 1)


def sample_passages(
    train: pd.DataFrame, model: str, n: int = 2, random_state: int = 0
) -> list[str]:
    """Random passages from a given model, for eyeballing its style."""
    subset = train[train[TARGET_COL] == model]
    return subset[TEXT_COL].sample(n=min(n, len(subset)), random_state=random_state).tolist()


def same_prompt_responses(
    train: pd.DataFrame, prompt_id: str | None = None, random_state: int = 0
) -> dict[str, str]:
    """One prompt answered by as many models as possible — compare styles
    side by side. Picks a random well-covered prompt if none given."""
    per_prompt_models = train.groupby(PROMPT_COL)[TARGET_COL].nunique()
    if prompt_id is None:
        candidates = per_prompt_models[per_prompt_models == per_prompt_models.max()]
        prompt_id = np.random.default_rng(random_state).choice(candidates.index)
    subset = train[train[PROMPT_COL] == prompt_id]
    return dict(zip(subset[TARGET_COL], subset[TEXT_COL], strict=False))


def distinctive_tokens(
    train: pd.DataFrame, top_n: int = 12, min_df: int = 5
) -> dict[str, pd.DataFrame]:
    """Most characteristic unigrams per model via one-vs-rest chi-squared.

    Returns ``{model: DataFrame[token, score]}`` — a quick read on each
    model's vocabulary tics (hedging words, formatting habits, etc.).
    """
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.feature_selection import chi2

    vec = CountVectorizer(
        lowercase=True,
        ngram_range=(1, 1),
        min_df=min_df,
        stop_words="english",
        token_pattern=r"(?u)\b[a-zA-Z'][a-zA-Z'-]+\b",
    )
    X = vec.fit_transform(train[TEXT_COL].fillna(""))
    y = train[TARGET_COL]
    vocab = np.array(vec.get_feature_names_out())

    out: dict[str, pd.DataFrame] = {}
    for model in LABELS:
        binary = (y == model).astype(int).values
        scores, _ = chi2(X, binary)
        top = np.argsort(scores)[::-1][:top_n]
        out[model] = pd.DataFrame({"token": vocab[top], "score": scores[top]})
    return out


# ---------------------------------------------------------------------------
# Verbal fingerprints (log-odds z-scores)
# ---------------------------------------------------------------------------


def log_odds_fingerprints(
    train: pd.DataFrame,
    top_n: int = 12,
    min_count: int = 20,
    exclude: set[str] | None = None,
) -> dict[str, pd.DataFrame]:
    """Per-model log-odds fingerprint: the words a model actually reaches for.

    The z-score column says how strongly a word belongs to that model
    relative to the other seventeen — which is what separates a *habit* from
    a *topic*.
    """
    texts = train[TEXT_COL].to_numpy(dtype=object)
    labels = train[TARGET_COL].to_numpy(dtype=object)
    return distinctive_profile(
        texts,
        labels,
        LABELS,
        top_n=top_n,
        min_count=min_count,
        exclude=exclude if exclude is not None else set(STOPWORDS),
    )


def family_fingerprints(
    train: pd.DataFrame, top_n: int = 12, min_count: int = 20
) -> dict[str, pd.DataFrame]:
    """Per-lab log-odds fingerprint.

    Measured on the train set this is much stronger than the per-model
    fingerprint: labs have a register, individual siblings do not.
    """
    texts = train[TEXT_COL].to_numpy(dtype=object)
    labels = train[TARGET_COL].to_numpy(dtype=object)
    families = sorted(set(PROVIDERS.values()))
    return {
        family: family_distinctive_words(
            texts, labels, PROVIDERS, family, top_n=top_n, min_count=min_count
        )
        for family in families
    }


def repeated_phrases(train: pd.DataFrame, model: str, top_n: int = 15, n: int = 2) -> pd.DataFrame:
    """The most repeated n-grams of a single model.

    This is the "does it have verbal tics?" question asked in the crudest
    possible way: which word sequences keep coming back?
    """
    from sklearn.feature_extraction.text import CountVectorizer

    subset = train[train[TARGET_COL] == model]
    vec = CountVectorizer(
        ngram_range=(n, n),
        lowercase=True,
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z'-]+\b",
    )
    counts = vec.fit_transform(subset[TEXT_COL].fillna(""))
    totals = np.asarray(counts.sum(axis=0)).ravel()
    vocab = vec.get_feature_names_out()
    order = np.argsort(totals)[::-1][:top_n]
    return pd.DataFrame({"phrase": vocab[order], "count": totals[order].astype(int)})


def sentence_openers(
    train: pd.DataFrame, model: str, top_n: int = 12, n_words: int = 3
) -> pd.DataFrame:
    """How a model *starts* its sentences.

    The opening of a sentence is one of the least topic-dependent habits a
    writer has: "It is worth noting that…", "There is a simple way to…",
    "Let me be clear:…".  Counting the first *n* words of every sentence
    turns that habit into a countable, comparable statistic.
    """
    import re as _re

    subset = train[train[TARGET_COL] == model]
    counter: dict[str, int] = {}
    for text in subset[TEXT_COL].fillna(""):
        for sentence in _re.split(r"(?<=[.!?])\s+", text):
            words = _re.findall(r"[A-Za-z][A-Za-z'-]+", sentence.lower())
            if len(words) < n_words:
                continue
            opener = " ".join(words[:n_words])
            counter[opener] = counter.get(opener, 0) + 1
    frame = pd.DataFrame({"opener": list(counter), "count": list(counter.values())}).sort_values(
        "count", ascending=False
    )
    return frame.head(top_n).reset_index(drop=True)


def opening_fingerprint(
    train: pd.DataFrame, top_n: int = 10, n_words: int = 3
) -> dict[str, pd.DataFrame]:
    """Sentence openers per model, restricted to openers used at least twice."""
    out: dict[str, pd.DataFrame] = {}
    for model in LABELS:
        frame = sentence_openers(train, model, top_n=top_n, n_words=n_words)
        out[model] = frame[frame["count"] >= 2].reset_index(drop=True)
    return out


# ---------------------------------------------------------------------------
# Recovered prompt groups
# ---------------------------------------------------------------------------


def recovered_group_report(train: pd.DataFrame, k: int = 3, cut: float = 0.95) -> dict:
    """How well does the inferred grouping recover the true prompt groups?"""
    texts = train[TEXT_COL].to_numpy(dtype=object)
    truth = train[PROMPT_COL].to_numpy(dtype=object)
    groups = infer_groups(texts, k=k, cut=cut)
    return group_report(groups, truth)


def view_examples(passage: str, max_chars: int = 240) -> dict[str, str]:
    """What each linguistic view does to one passage — the quickest way to
    see why one view is topic-free and another is not."""
    return {
        name: view_text(name, passage)[:max_chars]
        for name in ("char", "word", "shape", "funcon", "punct")
        if name in VIEWS
    }
# %% module: plots
"""Plotting helpers for exploration.

Each function returns a matplotlib ``(fig, ax)`` so notebooks can further
customize, and optionally saves to ``outputs/``. matplotlib/seaborn are
imported inside functions to keep imports light.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from .config import LABELS, PROVIDERS


def _finish(fig, ax, save_path: str | Path | None):
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=140, bbox_inches="tight")
    return fig, ax


def _provider_colors(models: list[str]) -> list[str]:
    palette = {
        "OpenAI": "#10a37f",
        "Google": "#4285f4",
        "Anthropic": "#d97757",
        "Qwen": "#7c3aed",
        "ZAI": "#0891b2",
        "Moonshot AI": "#e11d48",
    }
    return [palette.get(PROVIDERS.get(m, ""), "#888888") for m in models]


def plot_label_distribution(dist: pd.DataFrame, save_path: str | Path | None = None):
    """Bar chart of passages per model, coloured by provider.

    *dist* is the output of :func:`ntl.explore.label_distribution`.
    """
    import matplotlib.pyplot as plt

    d = dist.sort_values("n_passages", ascending=True)
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(d["model"], d["n_passages"], color=_provider_colors(list(d["model"])))
    for y, (n, share) in enumerate(zip(d["n_passages"], d["share"], strict=False)):
        ax.text(n + 2, y, f"{n}  ({share:.1%})", va="center", fontsize=8)
    ax.set_xlabel("passages in train.csv")
    ax.set_title("Class balance across the 18 models")
    ax.margins(x=0.15)
    return _finish(fig, ax, save_path)


def plot_prompt_coverage(pstats: pd.DataFrame, save_path: str | Path | None = None):
    """Histogram of distinct models per prompt (how complete each prompt is)."""
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(pstats["n_models"], bins=range(1, 20), color="#4c72b0", edgecolor="white")
    ax.set_xlabel("distinct models answering the prompt")
    ax.set_ylabel("number of prompts")
    ax.set_title("Prompt coverage: how many of the 18 models answered each prompt")
    return _finish(fig, ax, save_path)


def plot_length_by_model(train: pd.DataFrame, save_path: str | Path | None = None):
    """Box plot of word counts per model (competition says 40-250 words)."""
    import matplotlib.pyplot as plt
    import seaborn as sns

    fig, ax = plt.subplots(figsize=(11, 7))
    order = LABELS[::-1]
    d = train.assign(n_words=train["text"].str.split().str.len())
    sns.boxplot(
        data=d,
        y="model",
        x="n_words",
        order=order,
        hue="model",
        hue_order=order,
        palette=_provider_colors(order),
        legend=False,
        ax=ax,
    )
    ax.set_title("Word-count distribution per model")
    ax.set_xlabel("words per passage")
    ax.set_ylabel("")
    return _finish(fig, ax, save_path)


def plot_feature_heatmap(zscores: pd.DataFrame, save_path: str | Path | None = None):
    """Heatmap of z-scored style features per model (rows = models)."""
    import matplotlib.pyplot as plt
    import seaborn as sns

    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(
        zscores,
        cmap="RdBu_r",
        center=0,
        annot=False,
        cbar_kws={"label": "z-score vs other models"},
        ax=ax,
    )
    ax.set_title("Stylistic fingerprint per model (mean features, z-scored)")
    ax.set_xlabel("")
    ax.set_ylabel("")
    return _finish(fig, ax, save_path)


def plot_length_scatter(train: pd.DataFrame, save_path: str | Path | None = None):
    """Words vs characters, coloured by provider — sanity check on length."""
    import matplotlib.pyplot as plt

    d = train.assign(
        n_words=train["text"].str.split().str.len(),
        n_chars=train["text"].str.len(),
    )
    fig, ax = plt.subplots(figsize=(8, 6))
    for provider in sorted(set(PROVIDERS.values())):
        models = [m for m, p in PROVIDERS.items() if p == provider]
        sub = d[d["model"].isin(models)]
        ax.scatter(sub["n_words"], sub["n_chars"], s=8, alpha=0.4, label=provider)
    ax.set_xlabel("words")
    ax.set_ylabel("characters")
    ax.set_title("Passage length: words vs characters")
    ax.legend(title="provider", fontsize=8)
    return _finish(fig, ax, save_path)


def plot_distinctive_tokens(
    tokens: dict[str, pd.DataFrame],
    models: list[str] | None = None,
    save_path: str | Path | None = None,
):
    """Grid of top distinctive tokens per model (one small bar chart each)."""
    import matplotlib.pyplot as plt

    models = models or list(tokens.keys())
    ncols = 3
    nrows = -(-len(models) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 2.6 * nrows))
    axes = axes.ravel()
    for ax, model in zip(axes, models, strict=False):
        d = tokens[model].iloc[::-1]
        ax.barh(d["token"], d["score"], color="#4c72b0")
        ax.set_title(model, fontsize=10)
        ax.tick_params(labelsize=8)
    for ax in axes[len(models) :]:
        ax.axis("off")
    fig.suptitle("Most distinctive unigrams per model (one-vs-rest chi2)", y=1.001)
    fig.tight_layout()
    return _finish(fig, axes, save_path)


def plot_log_odds_fingerprints(
    profiles: dict[str, pd.DataFrame],
    models: list[str] | None = None,
    save_path: str | Path | None = None,
):
    """Grid of per-model verbal fingerprints (log-odds z-score).

    Each panel shows the words that model uses more than the other
    seventeen, ranked by effect size.  A long, confident bar list means a
    real habit; a short one means the model has no lexical tell.
    """
    import matplotlib.pyplot as plt

    models = models or list(profiles.keys())
    ncols = 3
    nrows = -(-len(models) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 2.8 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, model in zip(axes, models, strict=False):
        d = profiles[model]
        if d.empty:
            ax.axis("off")
            continue
        d = d.iloc[::-1]
        colours = _provider_colors([model]) * len(d)
        ax.barh(d["word"], d["z"], color=colours)
        ax.axvline(0, color="#999999", linewidth=0.8)
        ax.set_title(model, fontsize=10)
        ax.tick_params(labelsize=8)
    for ax in axes[len(models) :]:
        ax.axis("off")
    fig.suptitle(
        "Verbal fingerprint per model — log-odds z-score vs the other 17",
        y=1.001,
    )
    fig.tight_layout()
    return _finish(fig, axes, save_path)


def plot_family_fingerprints(
    profiles: dict[str, pd.DataFrame], save_path: str | Path | None = None
):
    """One panel per lab: the words that whole family favours."""
    import matplotlib.pyplot as plt

    families = list(profiles.keys())
    ncols = 3
    nrows = -(-len(families) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 2.8 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, family in zip(axes, families, strict=False):
        d = profiles[family]
        if d.empty:
            ax.axis("off")
            continue
        d = d.iloc[::-1]
        ax.barh(d["word"], d["z"], color="#4c72b0")
        ax.axvline(0, color="#999999", linewidth=0.8)
        ax.set_title(family, fontsize=10)
        ax.tick_params(labelsize=8)
    for ax in axes[len(families) :]:
        ax.axis("off")
    fig.suptitle("Register per lab — log-odds z-score vs the other labs", y=1.001)
    fig.tight_layout()
    return _finish(fig, axes, save_path)


def plot_group_sizes(
    sizes: np.ndarray,
    expected: int | None = None,
    save_path: str | Path | None = None,
):
    """Histogram of recovered group sizes.

    The dashed line marks the design size (18 passages per prompt): if the
    recovered groups cluster around it, the transductive grouping has found
    the prompt structure.
    """
    import matplotlib.pyplot as plt

    sizes = np.asarray(sizes)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    bins = np.arange(0.5, max(sizes.max() + 1.5, 20.5), 1.0)
    ax.hist(sizes, bins=bins, color="#4c72b0", edgecolor="white")
    if expected is not None:
        ax.axvline(
            expected + 0.5,
            color="#c44e52",
            linestyle="--",
            linewidth=1.4,
            label=f"design size ({expected} passages per prompt)",
        )
        ax.legend(fontsize=9)
    ax.set_xlabel("passages in recovered group")
    ax.set_ylabel("number of groups")
    ax.set_title("Recovered prompt groups: size distribution")
    return _finish(fig, ax, save_path)


def plot_ablation(
    steps: list[tuple[str, float]],
    baseline: float | None = None,
    save_path: str | Path | None = None,
):
    """Waterfall chart of what each pipeline layer is worth.

    *steps* is ``[(label, balanced_accuracy), ...]`` in the order the layers
    are applied; the bars show the running score and the annotations the
    gain of each layer.
    """
    import matplotlib.pyplot as plt

    labels = [s[0] for s in steps]
    values = [s[1] for s in steps]
    fig, ax = plt.subplots(figsize=(10, 5))

    ax.bar(range(len(values)), values, color="#4c72b0", width=0.62)
    for i, value in enumerate(values):
        ax.text(i, value + 0.006, f"{value:.3f}", ha="center", fontsize=9)
        if i > 0:
            delta = value - values[i - 1]
            ax.text(
                i,
                value * 0.5,
                f"{delta:+.3f}",
                ha="center",
                fontsize=9,
                color="#c44e52" if delta < 0 else "#2a7f62",
                fontweight="bold",
            )
    if baseline is not None:
        ax.axhline(
            baseline,
            color="#999999",
            linestyle=":",
            linewidth=1.2,
            label=f"chance ({baseline:.3f})",
        )
        ax.legend(fontsize=9)

    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=9)
    ax.set_ylabel("group-CV balanced accuracy")
    ax.set_ylim(0, max(values) * 1.18)
    ax.set_title("What each layer of the stack is worth")
    return _finish(fig, ax, save_path)


def plot_view_examples(examples: dict[str, str], save_path: str | Path | None = None):
    """Show what each linguistic view does to the same passage."""
    import textwrap

    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(len(examples), 1, figsize=(13, 2.1 * len(examples)))
    axes = np.atleast_1d(axes)
    for ax, (name, text) in zip(axes, examples.items(), strict=False):
        ax.axis("off")
        ax.set_title(name, loc="left", fontsize=10, fontweight="bold")
        ax.text(0, 0.9, textwrap.fill(text, 150), fontsize=8, va="top", family="monospace")
    fig.suptitle("One passage, five views", y=1.0)
    fig.tight_layout()
    return _finish(fig, axes, save_path)
# %% module: baseline
"""Classical baselines for authorship attribution.

The metric is **balanced accuracy** (mean per-class recall), so we evaluate
with that and use ``class_weight='balanced'``. Two complementary views:

* word n-grams  — vocabulary / phrasing fingerprints
* char n-grams  — punctuation, spacing, formatting fingerprints (strong for
  authorship attribution, and robust to the unseen-prompt shift)

A word of warning about the *evaluation*: random cross-validation does not
flatter this problem, it makes it a *different and much harder* task.  Every
prompt is answered by all 18 models, so a random split puts ~13.6 of each
validation passage's siblings into the training fold — each carrying a
*different* label.  The classifier is then asked to map one piece of
content to 18 different authors, and the cheapest way to fit that is to
ignore the content — and with it, the style.  The score collapses to 0.077
for words and 0.116 for characters, against 0.272 and 0.309 under
prompt-grouped folds.  Use :func:`grouped_cv_scores`.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline

from .config import PROMPT_COL, RANDOM_STATE, TARGET_COL, TEXT_COL


def _as_text_array(series: pd.Series) -> np.ndarray:
    """Return a plain numpy object array of strings.

    pandas 3.x defaults string columns to a pyarrow-backed dtype whose
    ``.values`` is not fancy-indexable by sklearn's parallel CV workers.
    """
    return series.fillna("").astype(object).to_numpy(dtype=object)


def build_word_pipeline(ngram_range: tuple[int, int] = (1, 2), min_df: int = 2) -> Pipeline:
    """TF-IDF word n-grams + logistic regression."""
    return Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    analyzer="word",
                    ngram_range=ngram_range,
                    min_df=min_df,
                    sublinear_tf=True,
                    strip_accents="unicode",
                ),
            ),
            (
                "clf",
                LogisticRegression(max_iter=2000, class_weight="balanced", C=4.0),
            ),
        ]
    )


def build_char_pipeline(
    ngram_range: tuple[int, int] = (2, 5),
    min_df: int = 3,
    max_features: int = 300_000,
) -> Pipeline:
    """TF-IDF char n-grams + logistic regression."""
    return Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=ngram_range,
                    min_df=min_df,
                    max_features=max_features,
                    sublinear_tf=True,
                    lowercase=False,
                ),
            ),
            (
                "clf",
                LogisticRegression(max_iter=2000, class_weight="balanced", C=4.0),
            ),
        ]
    )


def cross_val_scores(
    train: pd.DataFrame,
    pipeline: Pipeline | None = None,
    n_splits: int = 5,
    random_state: int = RANDOM_STATE,
) -> dict:
    """Random cross-validation — kept to demonstrate the *wrong task*.

    Every prompt is answered by all 18 models, so a random split puts ~13.6
    of each validation passage's siblings into the training fold, each with a
    *different* label.  The classifier is asked to map one piece of content
    to 18 different authors, and the cheapest way to fit that is to ignore
    the content — and with it, the style.  The score collapses to 0.077 for
    words and 0.116 for characters, against 0.272 and 0.309 under
    prompt-grouped folds.  Use :func:`grouped_cv_scores` for anything you
    intend to believe.
    """
    pipeline = pipeline or build_word_pipeline()
    X = _as_text_array(train[TEXT_COL])
    y = train[TARGET_COL].to_numpy(dtype=object)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof = cross_val_predict(pipeline, X, y, cv=cv, n_jobs=-1)
    return {
        "balanced_accuracy": balanced_accuracy_score(y, oof),
        "confusion": confusion_matrix(y, oof, labels=sorted(np.unique(y))),
        "labels": sorted(np.unique(y)),
        "oof": oof,
    }


def grouped_cv_scores(
    train: pd.DataFrame,
    pipeline: Pipeline | None = None,
    n_splits: int = 5,
    prompt_col: str = PROMPT_COL,
) -> dict:
    """Prompt-grouped cross-validation — the honest estimate.

    All the answers to one prompt land in the same fold, so the model can
    never learn a prompt's content from its siblings.
    """
    pipeline = pipeline or build_word_pipeline()
    X = _as_text_array(train[TEXT_COL])
    y = train[TARGET_COL].to_numpy(dtype=object)
    groups = train[prompt_col].to_numpy(dtype=object)
    cv = GroupKFold(n_splits=n_splits)
    oof = np.empty(len(y), dtype=object)
    for train_idx, valid_idx in cv.split(X, y, groups):
        pipeline.fit(X[train_idx], y[train_idx])
        oof[valid_idx] = pipeline.predict(X[valid_idx])
    return {
        "balanced_accuracy": balanced_accuracy_score(y, oof),
        "confusion": confusion_matrix(y, oof, labels=sorted(np.unique(y))),
        "labels": sorted(np.unique(y)),
        "oof": oof,
    }


def fit_predict(
    train: pd.DataFrame, test: pd.DataFrame, pipeline: Pipeline | None = None
) -> np.ndarray:
    """Fit on train, predict labels for test."""
    pipeline = pipeline or build_word_pipeline()
    pipeline.fit(_as_text_array(train[TEXT_COL]), train[TARGET_COL].to_numpy(dtype=object))
    return pipeline.predict(_as_text_array(test[TEXT_COL]))
# %% module: submit
"""Submission construction and validation.

The competition expects a ``submission.parquet`` file with exactly the
columns ``id`` and ``model`` — one row per test id, no nulls, and only the
18 valid model names.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from .config import ID_COL, LABELS, TARGET_COL


def make_submission(test: pd.DataFrame, predictions) -> pd.DataFrame:
    """Build the submission frame from test ids + predictions."""
    preds = pd.Series(np.asarray(predictions).ravel(), index=test.index)
    return pd.DataFrame({ID_COL: test[ID_COL].values, TARGET_COL: preds.values})


def validate_submission(sub: pd.DataFrame, expected_ids) -> list[str]:
    """Return a list of problems (empty list == valid).

    Checks columns, row count, id uniqueness/coverage, nulls and label validity.
    """
    problems: list[str] = []
    expected = set(expected_ids)

    if list(sub.columns) != [ID_COL, TARGET_COL]:
        problems.append(f"columns must be ['{ID_COL}', '{TARGET_COL}'], got {list(sub.columns)}")

    if sub[TARGET_COL].isna().any():
        problems.append("submission contains null model values")

    dupes = sub[ID_COL].duplicated().sum()
    if dupes:
        problems.append(f"{dupes} duplicated id(s)")

    got, missing = set(sub[ID_COL]), expected - set(sub[ID_COL])
    if missing:
        problems.append(f"{len(missing)} test id(s) missing from submission")
    extra = got - expected
    if extra:
        problems.append(f"{len(extra)} unknown id(s) in submission")

    bad = set(sub[TARGET_COL].dropna()) - set(LABELS)
    if bad:
        problems.append(f"{len(bad)} invalid label(s), e.g. {sorted(bad)[:3]}")

    return problems


def write_submission(sub: pd.DataFrame, path: str | Path = "submission.parquet") -> Path:
    """Validate semantics then write parquet (Kaggle code-competition default)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sub.to_parquet(path, index=False)
    return path
'''


def _ntl_bootstrap():
    """Materialise the `ntl` package and its submodules from inline source."""
    chunks = _NTL_SOURCES.split("\n" + _NTL_MARKER)[1:]
    sources = {}
    for chunk in chunks:
        name, _, body = chunk.partition("\n")
        sources[name.strip()] = body

    pkg = _types.ModuleType("ntl")
    pkg.__path__ = []
    pkg.__version__ = "0.4.0"
    _sys.modules["ntl"] = pkg

    order = ['config', 'data', 'features', 'views', 'facets', 'stylometry', 'groups', 'decision', 'pipeline', 'measure', 'tracker', 'explore', 'plots', 'baseline', 'submit']
    for name in order:
        mod = _types.ModuleType("ntl." + name)
        mod.__package__ = "ntl"
        _sys.modules["ntl." + name] = mod
        setattr(pkg, name, mod)
    for name in order:
        exec(compile(sources[name], "ntl/" + name + ".py", "exec"),
             _sys.modules["ntl." + name].__dict__)


_ntl_bootstrap()


In [2]:
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from ntl import data, pipeline, submit, tracker
from ntl.config import LABELS

ON_KAGGLE = Path("/kaggle/input").exists()
WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
WORKING.mkdir(parents=True, exist_ok=True)

print("data locations:")
for name, paths in data.data_locations().items():
    print(f"  {name}: {paths or '(not found)'}")

data locations:
  competition_dirs: (not found)
  attached_datasets: (not found)
  local_data_dir: /Users/Jerry/Developer/kaggle/name-that-llm/data/raw


In [3]:
VARIANT = "v3"
HYPOTHESIS = "Neither block works alone: voice alone is 0.4386 and punctuation alone 0.4327, both neutral. The bet is that they are complementary — punctuation gives the classifier a clean cue, voice facets give it the register that makes the cue usable."

VIEWS = ('char', 'word', 'funcon')
FACETS = ('length', 'lexical', 'punctuation', 'person', 'hedging', 'opener')
FACET_WEIGHT = 1.0

# Local CV is train-only and therefore comparable across variants.  Three
# folds keeps each notebook inside a few minutes; the ranking is stable.
RUN_LOCAL_CV = True
LOCAL_CV_FOLDS = 3

CONFIG_HASH = tracker.config_hash(VIEWS, FACETS, FACET_WEIGHT)
print(f"{VARIANT}  hash={CONFIG_HASH}")
print(f"views  : {VIEWS}")
print(f"facets : {FACETS or '(none)'}")
print(f"weight : {FACET_WEIGHT}")
print()
print(HYPOTHESIS)

v3  hash=766916c9f3
views  : ('char', 'word', 'funcon')
facets : ('length', 'lexical', 'punctuation', 'person', 'hedging', 'opener')
weight : 1.0

Neither block works alone: voice alone is 0.4386 and punctuation alone 0.4327, both neutral. The bet is that they are complementary — punctuation gives the classifier a clean cue, voice facets give it the register that makes the cue usable.


In [4]:
train = data.load_train()
train_texts = data.as_text_array(train["text"])
train_labels = data.as_text_array(train["model"])
train_groups = data.as_text_array(train["prompt_id"])

raw_pool = data.load_in_style_pool()
print("raw pool:", data.describe_in_style_pool(raw_pool))

# Filter to the 18 labels, drop train duplicates, namespace pool prompt ids.
pool = data.prepare_training_pool(raw_pool, train_texts=train_texts)
print(f"usable pool: {len(pool)} rows")

if len(pool):
    texts = np.concatenate([train_texts, data.as_text_array(pool["text"])])
    labels = np.concatenate([train_labels, data.as_text_array(pool["model"])])
    group_ids = np.concatenate([train_groups, data.as_text_array(pool["prompt_id"])])
else:
    texts, labels, group_ids = train_texts, train_labels, train_groups

print(f"training on {len(texts)} passages ({len(train)} train + {len(pool)} pool)")
uniq, counts = np.unique(labels, return_counts=True)
print("labels:", dict(zip(uniq.tolist(), counts.tolist(), strict=True)))

raw pool: {'rows': 0, 'models': {}, 'label_coverage': 0.0}
usable pool: 0 rows
training on 2699 passages (2699 train + 0 pool)
labels: {'claude-fable-5': 149, 'claude-opus-4-6': 150, 'claude-opus-4-8': 150, 'claude-opus-5': 150, 'claude-sonnet-5': 150, 'gemini-3.1-pro-preview': 150, 'gemini-3.6-flash': 150, 'gemini-3.7-flash': 150, 'gemini-3.8-flash': 150, 'glm-5.3': 150, 'glm-5.3-flash': 150, 'gpt-5.5': 150, 'gpt-5.6-luna': 150, 'gpt-5.6-sol': 150, 'gpt-5.6-terra': 150, 'gpt-6-astra': 150, 'kimi-k3': 150, 'qwen3.8-max': 150}


In [5]:
if RUN_LOCAL_CV:
    cv_config = pipeline.StackConfig(
        views=VIEWS, facets=FACETS, facet_weight=FACET_WEIGHT
    )
    cv = pipeline.group_cv_score(
        train_texts, train_labels, train_groups, cv_config, n_splits=LOCAL_CV_FOLDS
    )
    local_cv = float(cv["balanced_accuracy"])
    local_folds = LOCAL_CV_FOLDS
    print(f"local grouped CV ({LOCAL_CV_FOLDS} folds): {local_cv:.4f}")
    print("folds:", [round(score, 4) for score in cv["fold_scores"]])
    print(f"chance: {1 / len(LABELS):.4f}")
else:
    local_cv, local_folds = None, None
    print("local CV skipped")

local grouped CV (3 folds): 0.4730
folds: [0.4844, 0.4644, 0.4702]
chance: 0.0556


In [6]:
config = pipeline.StackConfig(views=VIEWS, facets=FACETS, facet_weight=FACET_WEIGHT)
model = pipeline.StackModel(config)
model.fit(texts, labels, groups=group_ids)

test = data.load_test()
test_texts = data.as_text_array(test["text"])
predictions = model.predict(test_texts)
print(f"predicted {len(predictions)} passages ({predictions.dtype})")

submission = submit.make_submission(test, predictions)
problems = submit.validate_submission(submission, test["id"])
print("validation problems:", problems or "none")

if ON_KAGGLE:
    out = WORKING / "submission.parquet"
else:
    out = WORKING / f"submission_{VARIANT}.parquet"
    print("NOTE: the local test.csv is a 3-row stub — this file is not a real submission.")

submit.write_submission(submission, out)
print("written:", out)
print()
print(submission["model"].value_counts().to_string())

predicted 3 passages (object)
validation problems: none
NOTE: the local test.csv is a 3-row stub — this file is not a real submission.
written: outputs/submission_v3.parquet

model
gpt-5.6-luna        1
gemini-3.6-flash    1
gemini-3.7-flash    1


In [7]:
row = tracker.make_row(
    variant=VARIANT,
    views=VIEWS,
    facets=FACETS,
    facet_weight=FACET_WEIGHT,
    local_cv=local_cv,
    local_folds=local_folds,
    n_train=len(train),
    n_pool=len(pool),
    notes="voice + punctuation",
)
tracker.append_csv(WORKING / "tracker.csv", row)

print("TRACKER_JSON " + tracker.to_json(row))
print()
print("After submitting, record the public score locally with:")
print(f"  uv run python scripts/track.py add --variant {VARIANT} "
      f"--views {','.join(VIEWS)} --facets {','.join(FACETS)} "
      f"--local-cv {0.0 if local_cv is None else local_cv:.4f} --public-lb <score>")

TRACKER_JSON {"config_hash": "766916c9f3", "facet_weight": 1.0, "facets": "length+lexical+punctuation+person+hedging+opener", "local_cv": 0.473038, "local_folds": 3, "n_pool": 0, "n_train": 2699, "notes": "voice + punctuation", "public_lb": null, "run_at": "2026-10-26T12:46:12Z", "variant": "v3", "views": "char+word+funcon"}

After submitting, record the public score locally with:
  uv run python scripts/track.py add --variant v3 --views char,word,funcon --facets length,lexical,punctuation,person,hedging,opener --local-cv 0.4730 --public-lb <score>


## Reading the result

- **Local CV up, public LB up** — the facet family is real. Keep it, and try
  giving it more weight (``facet_weight`` > 1).
- **Local CV up, public LB flat or down** — the gain is prompt-specific. The
  block is fitting the 150 train prompts, not the authors. Drop the family.
- **Both flat** — the family is null on this corpus (structure and emoji are,
  because the passages contain no markdown or emoji). Record the null; do not
  re-run it.

Download `/kaggle/working/tracker.csv` after each run and merge it:

```bash
uv run python scripts/track.py merge ~/Downloads/tracker.csv
uv run python scripts/track.py report
```